# Praktikum 5 — Pipeline Data Berita: dari Ingestion hingga MapReduce

| Keterangan | Isi |
|---|---|
| Mata Kuliah | Praktikum Big Data (DIF61142) |
| Program Studi / Semester | S1 Informatika / Semester 5 |
| Judul Praktikum | Pipeline data berita: dari ingestion, quality, storage, hingga MapReduce |
| Dosen | Afdhal Dinilhak & Ajeng Dwi Asti |
| Platform | Google Colab (runtime CPU) |
| Nama / NIM | Irgi Fatihul Ihsan / 2311533010 |

**Pertanyaan yang dijawab:** apa yang dibahas portal berita selama awal 2020, dan apakah topiknya berbeda antar-kategori?

**Format notebook ini:** setiap langkah terdiri dari *judul kecil → kode → interpretasi kode*. Kode H-1 s.d. H-13 sama dengan modul (bagian kunci sudah diisi). Bagian Analisis Hasil, Studi Kasus, Latihan, Tugas Mandiri, dan Pertanyaan Evaluasi dijawab dengan kode sebagai pembuktian.

> **Aturan jalan:** jalankan seluruh sel **berurutan dari atas**. Banyak sel memakai variabel dari sel sebelumnya.

## A. Peta Pipeline

| Tahap | Hasil | Langkah |
|---|---|---|
| Ingestion | `lapisan_mentah/` (JSONL) | H-3, H-4, H-5 |
| Quality & cleaning | DataFrame bersih | H-6 |
| Storage | `lapisan_terkurasi/` (Parquet berpartisi) | H-6 |
| Query & modeling | Tabel dan dokumen | H-7 |
| Processing (MapReduce) | Frekuensi kata | H-8 s.d. H-12 |
| Serving | `siap_pakai/` (Parquet dan database) | H-13 |

## G. Persiapan Praktikum
### Pasang pustaka

In [1]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

**Interpretasi:** sel ini memasang semua pustaka yang dipakai: `duckdb` (query SQL pada Parquet), `beautifulsoup4` + `lxml` (parsing HTML dan XML sitemap), `requests` (HTTP), `pyarrow` + `pandas` (Parquet dan DataFrame). Opsi `-q` membuat instalasi berjalan senyap.

## H-1. Peta Pipeline dan Permainan Kartu Kantin
### Simulasi permainan kartu dengan kode

Tiga kasir mencatat struk masing-masing. Tahap yang dimainkan: **map** (satu kartu per baris), **shuffle** (kartu semenu dikumpulkan ke satu meja), **reduce** (jumlahkan per meja), lalu ulangi dengan **combiner** (tiap kasir menjumlahkan dulu kartunya sendiri).

In [2]:
# Struk tiga kasir (sama dengan tabel di modul)
struk_kasir = {
    "Kasir A": [("nasi goreng", 1), ("es teh", 2), ("mie ayam", 1), ("es teh", 1)],
    "Kasir B": [("nasi goreng", 2), ("es teh", 1), ("es teh", 3), ("mie ayam", 1)],
    "Kasir C": [("mie ayam", 2), ("nasi goreng", 1), ("es teh", 1), ("nasi goreng", 1)],
}

# Cara 1: tanpa combiner -> setiap kartu (baris struk) pindah ke meja reducer
kartu_pindah_tanpa = sum(len(kartu) for kartu in struk_kasir.values())

# Reduce: jumlahkan per menu
total_menu = {}
for kartu in struk_kasir.values():
    for menu, jumlah in kartu:
        total_menu[menu] = total_menu.get(menu, 0) + jumlah

# Cara 2: dengan combiner -> tiap kasir menjumlahkan dulu per menu
kartu_pindah_dengan = 0
for nama, kartu in struk_kasir.items():
    lokal = {}
    for menu, jumlah in kartu:
        lokal[menu] = lokal.get(menu, 0) + jumlah
    kartu_pindah_dengan += len(lokal)
    print(f"{nama} menjumlahkan dulu ->", lokal)

print("\nHasil reduce       :", total_menu)
print("Kartu pindah tanpa combiner :", kartu_pindah_tanpa)
print("Kartu pindah dengan combiner:", kartu_pindah_dengan)

Kasir A menjumlahkan dulu -> {'nasi goreng': 1, 'es teh': 3, 'mie ayam': 1}
Kasir B menjumlahkan dulu -> {'nasi goreng': 2, 'es teh': 4, 'mie ayam': 1}
Kasir C menjumlahkan dulu -> {'mie ayam': 2, 'nasi goreng': 2, 'es teh': 1}

Hasil reduce       : {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4}
Kartu pindah tanpa combiner : 12
Kartu pindah dengan combiner: 9


**Interpretasi:** hasil reduce harus **nasi goreng 5, es teh 8, mie ayam 4** (total 17 porsi). Tanpa combiner ada **12** kartu yang berpindah meja, dengan combiner hanya **9**. Hasil akhirnya sama persis, tetapi yang dipindahkan lebih sedikit. Angka 12 dan 9 ini akan dicocokkan dengan kode pada H-8.

## H-2. Persiapan: Unduh Data dan Jalankan Portal Lokal
### Unduh CSV dan lihat ukurannya

In [3]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas :", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom  :", df_sumber.shape)
print("Kolom         :", list(df_sumber.columns))

Ukuran berkas : 16.1 MB
Baris, kolom  : (91017, 4)
Kolom         : ['date', 'url', 'title', 'category']


**Interpretasi:** dataset berupa satu CSV berukuran **16,1 MB**, **91.017 baris**, dengan kolom `date`, `url`, `title`, `category`. Dataset hanya berisi judul (bukan isi artikel) dan diunduh dari sumbernya untuk pembelajaran saja (repositori tidak mencantumkan lisensi; kutip sumbernya di laporan).

### Nyalakan portal berita lokal

In [4]:
import http.server, urllib.parse, html

HALAMAN_UKURAN = 100  # judul per halaman

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):  # matikan log
            pass

        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


**Interpretasi:** sel ini menyalakan *web server* mini di komputer kita (thread di latar belakang) yang menyajikan isi CSV sebagai halaman HTML `/berita?hal=N` berisi 100 judul per halaman, plus `robots.txt` yang melarang `/arsip-privat/`. Dengan 91.017 judul / 100 per halaman didapat **911 halaman**. Tujuannya agar kita berlatih crawling tanpa membebani situs sungguhan dan hasilnya sama untuk semua mahasiswa.

## H-3. Crawling Portal Lokal: Latihan Aman
### Baca dan patuhi `robots.txt`

In [5]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


**Interpretasi:** crawler yang baik membaca `robots.txt` lebih dulu. `RobotFileParser.can_fetch(UA, url)` menjawab apakah User-Agent kita boleh mengakses alamat itu. Hasilnya: `/berita?hal=1` **boleh** (`True`), `/arsip-privat/` **dilarang** (`False`). Aturan ini berlaku per host.

### Parser HTML dan crawler yang sopan

In [6]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


**Interpretasi:** `parse_halaman` memakai CSS selector: setiap judul dibungkus `article.item`; judul + tautan di `h2.judul a`, kategori di `span.kategori`, dan tanggal ISO di atribut `datetime` milik `<time>` (lebih aman daripada teks `02/26/2020` yang ambigu bulan/hari). `crawl_iter` memakai `yield from`, sehingga mengembalikan satu hasil demi satu (itulah yang memungkinkan crawler menjadi *source* pada H-5), serta punya *retry* 3 kali dan jeda antar-halaman. Dari 3 halaman didapat **300 judul** (3 × 100).

## H-4. Crawling Berita Sungguhan dari Sitemap detik.com
### Langkah 1 — Tulis parser sitemap (bagian kunci, sudah diisi)

In [7]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else None,  # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

**Interpretasi:** bagian kunci yang diisi: `find("title")` dan `find("publication_date")` (awalan namespace `news:` diabaikan saat mencari tag), kategori diambil dari **subdomain** URL (`m.group(1)` → `news`, `inet`, `sport`), dan tanggal diambil **10 karakter pertama** dari waktu terbit (`2026-10-29`). Entri tanpa `loc` atau judul dilewati agar satu data cacat tidak menghentikan seluruh proses. Isi `CDATA` terbaca sebagai teks biasa.

### Langkah 2 — Uji parser tanpa internet

In [8]:
# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
      <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


**Interpretasi:** parser diuji pada XML kecil sebelum menghubungi server. Hasilnya **3 entri**, kategori `news`, id `8729282 / 8729280 / 8729264`, tanggal `2026-10-29`. Kata kunci hanya ada pada entri pertama (`mahkamah konstitusi,mk,hukum,kepala daerah`), dan entri lain tetap terbaca dengan `kata_kunci` kosong (tidak error). Masukan `str` maupun `bytes` menghasilkan data yang sama (`True`).

### Langkah 3 — Fungsi pengambilan yang sopan

In [9]:
import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: GANTI_DENGAN_EMAIL_ANDA)"
DELAY = 2.0              # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7     # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20         # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}

def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content  # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

> ⚠️ **Sebelum menjalankan sel crawl di bawah:** ganti `GANTI_DENGAN_EMAIL_ANDA` pada `UA_DETIK` dengan email Anda, lalu jalankan ulang sel di atas.

**Interpretasi:** ada enam pengaman: (1) `robots.txt` dibaca **per host** dan disimpan di kamus `_robots` supaya hanya diunduh sekali per host; (2) hanya **3 sitemap** yang diambil; (3) maksimum **7 judul per sitemap** dan **20 judul total**; (4) **jeda 2 detik** antar-request; (5) `User-Agent` menyebut identitas kita; (6) berhenti seketika bila server membalas **403/429** agar tidak terus mendesak server. `r.content` (bytes) dipakai, bukan `r.text`, agar pengkodean dibaca dari deklarasi XML dan tidak ditebak.

### Langkah 4 — Jalankan dan periksa

In [10]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul :", len(cek))
    print("Judul kosong    :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat    :", cek["url"].duplicated().sum())
    print("Tanggal kosong  :", cek["tanggal"].isna().sum())
    print("Kategori        :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul : 20
Judul kosong    : 0
URL duplikat    : 0
Tanggal kosong  : 0
Kategori        : {'news': 7, 'inet': 7, 'sport': 6}
                                                                      judul kategori    tanggal
  Polri Jerat 20 Korporasi Tersangka Karhutla, Legislator PDIP: Hukum Berat     news 2026-10-30
          KPK Ungkap Perkara Cukai Seret PT Djarum Rugikan Negara Triliunan     news 2026-10-30
MBG Diusulkan Masuk Kurikulum, Komisi X DPR: Perlu Dikaji Serius Urgensinya     news 2026-10-30


**Interpretasi:** hasil yang diharapkan: **20 judul**, tanpa judul kosong dan tanpa URL duplikat, dengan kategori `news`, `inet`, `sport` (**7, 7, 6**). Isinya berbeda setiap hari sehingga tidak ada angka acuan tetap. Hasil disimpan di folder `contoh_crawl/`, terpisah dari `lapisan_mentah/` agar angka acuan langkah berikutnya tidak berubah.

**Catatan zona waktu:** pada uji dosen, entri terbaru bertanda `12:52:00+00:00` padahal waktu sebenarnya sekitar 12.55 WIB, jadi tanda zona waktunya kemungkinan keliru. Karena itu kode hanya mengambil **tanggal** (10 karakter pertama) dan tidak mengonversi zona waktu.

**Etika:** jangan jalankan sel ini berulang dan jangan menaikkan `BATAS_TOTAL` (lab sering memakai satu IP publik bersama). Ambil secukupnya, simpan hanya judul, URL, kategori, tanggal, dan kata kunci, dan pakai untuk pembelajaran saja.

## H-5. Channel ala Flume ke `lapisan_mentah`
### Source → channel → sink

In [11]:
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)  # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []
    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0

print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3], "...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 49.8 detik untuk 911 halaman


**Interpretasi:** crawler berperan sebagai **source**, `queue.Queue(maxsize=2000)` sebagai **channel**, dan fungsi `sink` sebagai penulis JSONL (satu berkas per 5.000 event). Source dan sink berjalan di thread terpisah; bila channel penuh, `put()` menunggu (*backpressure*) sehingga memori tidak membengkak. Pemeriksaan pertama pada setiap pipeline: **event masuk = event ditulis = 91.017**; angka yang berbeda berarti ada data hilang di tengah jalan. Hasilnya **19 berkas batch** (⌈91.017 / 5.000⌉ = 19), antrean tidak pernah melewati 2.000, dan ukuran mentah ± 20,7 MB. Waktu crawl bergantung komputer.

## H-6. Quality, Cleaning, dan Parquet Berpartisi
### Muat lapisan mentah dan periksa kualitas (bagian kunci, sudah diisi)

In [12]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))],
                   ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong": d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"   : jml_kata.between(3, 30),
        "kategori dikenal"  : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"     : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid" : d["url"].fillna("").str.contains(r"/d-\d+", regex=True),
        "judul tidak duplikat": ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)")

Baris mentah: 91,017
judul tidak kosong       gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
kategori dikenal         gagal:      0 ( 0.00%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)


**Interpretasi:** setiap aturan kualitas adalah ekspresi pandas yang menghasilkan `True/False` per baris. Bagian kunci yang diisi: batas judul **3–30 kata** (`between(3, 30)`), kategori harus ada di himpunan `KATEGORI_DIKENAL` (`isin`), dan URL artikel valid bila memuat pola `/d-<angka>`. `duplicated(keep=False)` menandai **semua** salinan, bukan hanya salinan kedua dan seterusnya.

Temuan kualitas data nyata: aturan **url artikel valid** gagal untuk **5.640** baris, tepat sebanyak seluruh baris kategori `inet` (scraper menyimpan `#` sebagai URL untuk satu kategori penuh). Aturan lain: judul 3–30 kata gagal 78 baris (0,09%) dan judul duplikat 235 baris (0,26%). Pilihan membuang, memperbaiki, atau mempertahankan baris cacat adalah keputusan analisis dan alasannya harus dicatat di laporan.

### Pembersihan data

In [13]:
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")  # kosong bila url rusak
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))
print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

Baris setelah dibersihkan: 90,818 | dibuang: 199


**Interpretasi:** pembersihan merapikan spasi, membuang judul duplikat (simpan yang pertama), membuang judul yang terlalu pendek/panjang, mengubah tanggal ke `datetime` dengan **format eksplisit**, serta membuat kolom `bulan` dan `id`. Baris `inet` **tidak dibuang**: kolom `url` dikosongkan (`NaN`) dan `id` dibiarkan kosong agar tidak dipakai sebagai kunci, sementara judulnya tetap berguna untuk analisis teks. Hasilnya **90.818 baris** (dibuang 199).

### Simpan sebagai Parquet berpartisi (bagian kunci: pilih kolom partisi)

In [14]:
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# BAGIAN KUNCI: bandingkan pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori = finance")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


**Interpretasi:** tiga pilihan kolom partisi dibandingkan: `kategori` → **9 file**, `bulan` → **6 file**, `kategori + bulan` → **54 file** (ukuran total hampir sama, ± 8,4 MB). Dipilih partisi **kategori**: jumlah berkasnya cukup sedikit untuk menghindari *small file problem*, pertanyaan kita menyangkut kategori, dan berkas-berkas ini nanti menjadi **input split MapReduce**. `kategori + bulan` hanya menghasilkan banyak berkas kecil tanpa keuntungan. *Partition pruning* terbukti: untuk `kategori = finance` hanya **1 dari 9** berkas yang dibaca.

## H-7. Query DuckDB dan Bentuk Dokumen
### Query tabel dan buat dokumen

In [15]:
import duckdb
con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


**Interpretasi:** DuckDB membaca seluruh folder Parquet sebagai satu tabel; kolom `kategori` diambil dari **nama folder** (*hive partitioning*). Jumlah judul per kategori: `news` 32.321, `hot` 16.290, `finance` 14.141, dan seterusnya sampai `sport` 2.433. Judul yang memuat "corona" **memuncak pada Maret 2020 (6.994)**; Juni hanya mencakup 13 hari pertama. Bentuk **tabel** cocok untuk menghitung dan mengelompokkan, sedangkan bentuk **dokumen** (JSON) cocok menyimpan satu artikel beserta atribut turunan (daftar kata/tag) tanpa mengubah skema tabel.

## H-8. Mini MapReduce: Kasus Kantin
### Tiga fungsi: map, shuffle, reduce, dan combiner

In [16]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],    # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],    # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"],  # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner :", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner : 12
Pasangan dikirim dengan combiner: 9


**Interpretasi:** tiga fungsi mewakili tiga tahap: `map_fn` mengubah baris menjadi `(key, value)`, `shuffle` mengelompokkan nilai berdasarkan key, `reduce_fn` menjumlahkan. Hasilnya **nasi goreng 5, es teh 8, mie ayam 4** (total 17), sama dengan permainan kartu H-1. Combiner menurunkan pasangan yang dikirim dari **12 menjadi 9** tanpa mengubah hasil akhir, sama dengan hitungan kartu pada H-1.

## H-9. Word Count Terdistribusi: Partisi sebagai Input Split
### Satu partisi Parquet = satu mapper

In [17]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")

print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")

top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


**Interpretasi:** satu **input split** = satu berkas partisi Parquet, jadi ada **9 mapper**, dan ukurannya sangat tidak seimbang (`news` 32 ribu judul vs `sport` 2 ribu). Seluruh mapper menghasilkan **825.601 pasangan** `(kata, 1)`; setelah shuffle ada **32.275 kata unik** (key). Kata teratas semua kata adalah kata fungsi (`di` 27.390, ...), sedangkan tanpa stopword kata teratas adalah **corona (17.239)**. Hasil MapReduce **identik** dengan `Counter` biasa (`True`), sehingga implementasinya terbukti benar.

Catatan: daftar stopword hanya berisi kata fungsi umum, sehingga `rp`, `ri`, `the` tetap muncul. Menentukan stopword untuk domain tertentu adalah keputusan analisis (dibahas pada Analisis Hasil no. 3).

## H-10. Combiner dan Jumlah Data yang Berpindah
### Ukur waktu 1 vs 2 proses, dengan dan tanpa combiner

In [18]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")
for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h):>20,}")
    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.827              825,601
1 proses, dengan combiner         0.947               74,009
2 proses, tanpa combiner          1.141              825,601
2 proses, dengan combiner         0.635               74,009

Hasil dengan combiner sama persis: True


**Interpretasi:** angka **pasangan dikirim stabil**: tanpa combiner **825.601**, dengan combiner **74.009** (turun ± 11 kali) dengan hasil akhir yang persis sama (`True`). Waktu bervariasi antar-komputer. Pada contoh modul, 2 proses tanpa combiner justru lebih lambat daripada 1 proses karena biaya memindahkan 825 ribu pasangan antar-proses lebih besar daripada kerja mapper-nya; dengan combiner, 2 proses sedikit lebih cepat. Pelajaran utama: **paralel tidak otomatis lebih cepat; yang mahal adalah memindahkan data.**

## H-11. Data Skew dan Beban Reducer
### Partitioner berbasis hash (CRC32) dan beban tiap reducer

In [19]:
R = 4  # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", "  ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%)  R1=170,066 (20.6%)  R2=190,727 (23.1%)  R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%)  R1= 18,267 (24.7%)  R2= 18,530 (25.0%)  R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%
   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


**Interpretasi:** `partitioner` memakai `zlib.crc32` (bukan `hash()` Python yang nilainya berubah antar-sesi) sehingga hasilnya sama di setiap komputer. Dengan 4 reducer, reducer terberat menerima **29,2%** pasangan (bagian adil 25%). Pada 64 reducer, terberat **4,28%** padahal bagian adilnya 1,56% (hampir tiga kali lipat), karena kata `di` sendirian sudah **3,3%** dari seluruh pasangan dan **satu key tidak bisa dipecah** ke banyak reducer. Jadi menambah reducer tidak menyelesaikan skew yang berasal dari satu key. Combiner membantu: beban empat reducer menjadi hampir rata (**24,7%–25,6%**) karena setiap kata hanya dikirim sekali per mapper.

## H-12. Hadoop Streaming
### Tulis berkas masukan, `mapper.py`, dan `reducer.py`

In [20]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')

print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


**Interpretasi:** Hadoop Streaming menjalankan mapper dan reducer sebagai program biasa yang membaca `stdin` dan menulis `stdout`, sehingga bahasa apa pun bisa dipakai. Berkas masukan berisi `kategori<TAB>judul` untuk **90.818 baris**. **Mapper** mencetak `kata<TAB>1`. **Reducer** mengandalkan masukan yang sudah **terurut per key**: selama key sama, jumlahnya ditambahkan; saat key berganti, hasil key sebelumnya dicetak.

### Uji dengan pipe Linux (`cat | mapper | sort | reducer`)

In [21]:
!head -n 3 masukan.tsv | python3 mapper.py
!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1

real	0m1.805s
user	0m1.967s
sys	0m0.065s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


**Interpretasi:** mapper diuji pada tiga baris pertama, lalu seluruh pipeline dijalankan. Perintah `sort` berperan sebagai **shuffle** (`LC_ALL=C` memastikan urutan berdasarkan byte, sama seperti pengurutan key di Hadoop). Hasilnya **32.275 baris** (sama dengan jumlah kata unik H-9), dan lima kata teratas: `di 27390`, `corona 17239`, `ini 10124`, `yang 8668`, `dan 6785`. Waktu (`real/user/sys`) berbeda tiap komputer.

### Buktikan hasil pipe sama dengan H-9

In [22]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


**Interpretasi:** `True` berarti mapper dan reducer gaya Hadoop Streaming menghasilkan frekuensi kata yang **persis sama** dengan versi Python di memori (H-9). Jadi logika yang sama dapat dipindahkan ke Hadoop sungguhan.

### (Opsional) Hadoop asli

Menjalankan mapper dan reducer yang sama pada Hadoop sungguhan (local mode) bersifat **opsional / nilai tambah**. Sel-nya saya letakkan di **bagian paling akhir notebook** agar tidak memperlambat atau menghentikan *Run all* (unduhan Hadoop ± 700 MB).

## H-13. Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman
### Key gabungan `(kategori, kata)` dengan combiner di dalam mapper

In [23]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))

for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)


**Interpretasi:** key diganti menjadi pasangan `(kategori, kata)` dengan stopword dibuang di sisi mapper; setiap mapper memproses satu partisi (satu kategori) dan combiner (`Counter`) dipasang di dalam mapper. Kata **corona** mendominasi hampir semua kategori (misalnya `news` 7.539, `health` 2.590, `finance` 2.470), sedangkan kategori `food` (makanan, makan, foto, galeri, enak) dan `oto` (mobil, motor) paling berbeda: topiknya mengikuti tema kategorinya.

### Ekspor ala Sqoop ke SQLite

In [24]:
import sqlite3

SIAP = Path("siap_pakai"); SIAP.mkdir(exist_ok=True)

baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
                    WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


**Interpretasi:** hasil MapReduce disimpan ke `siap_pakai/kata_per_kategori.parquet` dan **diekspor** ke database SQLite `laporan.db` dengan `to_sql`, mensimulasikan `sqoop export` (hasil pemrosesan dikirim ke database relasional untuk laporan). Terdapat **73.601 baris** `(kategori, kata, jumlah)`. Query contoh pada kategori `health` mengembalikan tiga kata teratas: `corona (2.590)`, `virus (1.040)`, `pasien (471)`. Seluruh alur lengkap: portal → channel → bersih → Parquet berpartisi → MapReduce → database laporan.

## K. Output yang Diharapkan: Pemeriksaan Otomatis
### Cocokkan angka "tetap" modul dengan hasil notebook Anda

In [25]:
gagal_url = int((~aturan_kualitas(mentah)["url artikel valid"]).sum())
beban_4 = max(beban(hasil_map)) / sum(beban(hasil_map)) * 100

pemeriksaan = [
    ("Baris CSV (H-2)",                         df_sumber.shape,                        (91017, 4)),
    ("Halaman di-crawl (H-5)",                  N_HALAMAN,                              911),
    ("Event masuk = ditulis (H-5)",             (statistik["masuk"], statistik["keluar"]), (91017, 91017)),
    ("File batch lapisan_mentah (H-5)",         statistik["file"],                      19),
    ("Gagal aturan url artikel valid (H-6)",    gagal_url,                              5640),
    ("Baris setelah dibersihkan (H-6)",         len(bersih),                            90818),
    ("File partisi kategori (H-6)",             len(splits),                            9),
    ("Partition pruning finance (H-6)",         (len(hanya), len(semua)),               (1, 9)),
    ("Hasil kantin (H-8)",                      hasil,                                  {"nasi goreng": 5, "es teh": 8, "mie ayam": 4}),
    ("Pasangan dari semua mapper (H-9)",        total_pasangan,                         825601),
    ("Kata unik (H-9)",                         len(hasil_semua),                       32275),
    ("Pasangan dengan combiner (H-10)",         sum(len(x) for x in h2),                74009),
    ("Beban reducer terberat, 4 reducer (H-11)", round(beban_4, 2),                     29.17),
    ("Hasil pipe = hasil H-9 (H-12)",           hasil_pipe == hasil_semua,              True),
]
print(f"{'pemeriksaan':<44}{'hasil Anda':>28}  {'status'}")
for nama, nilai, harapan in pemeriksaan:
    print(f"{nama:<44}{str(nilai):>28}  {'OK' if nilai == harapan else 'BEDA (harapan: ' + str(harapan) + ')'}")

pemeriksaan                                                   hasil Anda  status
Baris CSV (H-2)                                               (91017, 4)  OK
Halaman di-crawl (H-5)                                               911  OK
Event masuk = ditulis (H-5)                               (91017, 91017)  OK
File batch lapisan_mentah (H-5)                                       19  OK
Gagal aturan url artikel valid (H-6)                                5640  OK
Baris setelah dibersihkan (H-6)                                    90818  OK
File partisi kategori (H-6)                                            9  OK
Partition pruning finance (H-6)                                   (1, 9)  OK
Hasil kantin (H-8)                          {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4}  OK
Pasangan dari semua mapper (H-9)                                  825601  OK
Kata unik (H-9)                                                    32275  OK
Pasangan dengan combiner (H-10)                       

**Interpretasi:** sel ini membandingkan setiap angka bersifat **tetap** pada tabel "Output yang Diharapkan" modul dengan hasil notebook ini. Semua baris harus berstatus `OK`. Waktu crawl, waktu mapper, dan waktu pipe memang bervariasi antar-komputer sehingga tidak diperiksa.

---
# L. Analisis Hasil
Setiap jawaban terdiri dari: **judul kecil → kode pembuktian → interpretasi** dengan merujuk angka dari output.

## L-1. Mengapa 2 proses tanpa combiner bisa lebih lambat daripada 1 proses?
**Pertanyaan:** Pada H-10, mengapa 2 proses tanpa combiner bisa lebih lambat daripada 1 proses? Apa yang berubah ketika combiner dipasang?

### Kode: ukur data yang harus dikirim antar-proses, lalu ukur ulang waktunya

In [26]:
import pickle

# (a) Hasil mapper yang dikembalikan Pool harus diserialisasi (pickle) lalu dikirim ke proses induk,
#     mirip data yang dikirim lewat jaringan pada tahap shuffle.
pasangan_tanpa, pasangan_dengan = sum(len(x) for x in h), sum(len(x) for x in h2)
byte_tanpa, byte_dengan = len(pickle.dumps(h)), len(pickle.dumps(h2))
print(f"{'':<18}{'pasangan':>12}{'ukuran pickle (MB)':>22}")
print(f"{'tanpa combiner':<18}{pasangan_tanpa:>12,}{byte_tanpa/1e6:>22.2f}")
print(f"{'dengan combiner':<18}{pasangan_dengan:>12,}{byte_dengan/1e6:>22.2f}")
print(f"\nPenurunan jumlah pasangan : {pasangan_tanpa/pasangan_dengan:.1f}x")
print(f"Penurunan ukuran serialisasi: {byte_tanpa/byte_dengan:.1f}x")

# (b) Waktu (median tiga kali jalan) 1 proses vs 2 proses, dengan dan tanpa combiner
def median_waktu(fn, proses, ulang=3):
    w = sorted(jalankan(fn, splits, proses)[1] for _ in range(ulang))
    return w[1]

print(f"\nCPU tersedia: {os.cpu_count()}")
print(f"{'skenario':<28}{'1 proses (s)':>14}{'2 proses (s)':>14}{'rasio 2/1':>11}")
for nama, fn in (("tanpa combiner", mapper_split), ("dengan combiner", mapper_combiner)):
    t1, t2 = median_waktu(fn, 1), median_waktu(fn, 2)
    print(f"{nama:<28}{t1:>14.3f}{t2:>14.3f}{t2/t1:>11.2f}")

                      pasangan    ukuran pickle (MB)
tanpa combiner         825,601                 10.35
dengan combiner         74,009                  0.87

Penurunan jumlah pasangan : 11.2x
Penurunan ukuran serialisasi: 12.0x

CPU tersedia: 2
skenario                      1 proses (s)  2 proses (s)  rasio 2/1
tanpa combiner                       0.817         1.600       1.96
dengan combiner                      0.558         0.610       1.09


**Interpretasi:**
- Kerja mapper itu sendiri **kecil** (membaca Parquet dan tokenisasi 90 ribu judul kurang dari satu detik). Pada `Pool.map`, setiap mapper harus **mengembalikan hasilnya ke proses induk**, dan hasil itu diserialisasi (*pickle*), dikirim lewat pipa antar-proses, lalu dibaca kembali. Tanpa combiner yang dikirim adalah **825.601 pasangan** `(kata, 1)` (± 10 MB setelah diserialisasi); biaya memindahkan data ini lebih besar daripada penghematan waktu dari bekerja paralel, sehingga 2 proses bisa **lebih lambat** daripada 1 proses (1 proses tidak perlu memindahkan apa pun antar-proses).
- Ketika combiner dipasang, setiap mapper menjumlahkan dulu di sisi lokal dan hanya mengirim satu pasangan per kata unik: **74.009 pasangan** (turun ± 11 kali; ukuran serialisasi ikut turun ± 12 kali, dari ± 10 MB menjadi < 1 MB, seperti terlihat pada tabel pertama). Biaya pemindahan data menyusut sehingga paralelisme mulai terlihat manfaatnya, walaupun selisihnya kecil karena datanya sendiri kecil.
- Pelajaran: **paralel tidak otomatis lebih cepat; yang mahal adalah memindahkan data (shuffle)**. Waktu pada tabel kedua bervariasi antar-komputer (jumlah CPU Colab biasanya hanya 2), tetapi jumlah pasangan selalu sama.

## L-2. Mengapa menambah reducer (4 → 64) tidak meratakan beban?
**Pertanyaan:** Pada H-11, mengapa menambah reducer dari 4 menjadi 64 tidak membuat beban reducer terberat mendekati bagian yang adil? Bagaimana combiner mengubah gambarnya?

### Kode: bandingkan beban reducer terberat dengan dan tanpa combiner

In [27]:
pasangan_combiner = [list(c.items()) for c in h2]
total_tanpa = sum(len(h_) for h_ in hasil_map)
total_dengan = sum(len(h_) for h_ in pasangan_combiner)

print(f"{'reducer':>8} {'adil':>7} | {'terberat tanpa comb.':>21} {'x adil':>7} | {'terberat dengan comb.':>22} {'x adil':>7}")
for r in (4, 16, 64):
    b1, b2 = beban(hasil_map, r), beban(pasangan_combiner, r)
    adil = 100 / r
    p1, p2 = max(b1) / sum(b1) * 100, max(b2) / sum(b2) * 100
    print(f"{r:>8} {adil:>6.2f}% | {p1:>20.2f}% {p1/adil:>6.2f}x | {p2:>21.2f}% {p2/adil:>6.2f}x")

# Satu key tidak bisa dipecah: batas bawah beban reducer = bagian key terbesar
kata_top, n_top = collections.Counter(hasil_semua).most_common(1)[0]
print(f"\nKey terbesar tanpa combiner : '{kata_top}' = {n_top:,} pasangan = {n_top/total_tanpa*100:.2f}% dari total")
muncul_dengan = sum(1 for c in h2 if c[kata_top] > 0)   # tiap mapper mengirim satu pasangan untuk key ini
print(f"Key yang sama dengan combiner: '{kata_top}' hanya {muncul_dengan} pasangan "
      f"(satu per mapper) = {muncul_dengan/total_dengan*100:.3f}% dari total")

 reducer    adil |  terberat tanpa comb.  x adil |  terberat dengan comb.  x adil
       4  25.00% |                29.17%   1.17x |                 25.55%   1.02x
      16   6.25% |                 8.82%   1.41x |                  6.54%   1.05x
      64   1.56% |                 4.28%   2.74x |                  1.86%   1.19x

Key terbesar tanpa combiner : 'di' = 27,390 pasangan = 3.32% dari total
Key yang sama dengan combiner: 'di' hanya 9 pasangan (satu per mapper) = 0.012% dari total


**Interpretasi:**
- Bagian yang adil memang mengecil mengikuti `100/R` (25% → 1,56%), tetapi reducer terberat **tidak bisa lebih kecil daripada key terbesar**. Tanpa combiner, kata `di` sendirian = **3,3%** dari seluruh pasangan, dan karena partitioner mengirim **semua pasangan satu key ke satu reducer yang sama**, reducer pemegang `di` minimal menerima 3,3%. Pada 64 reducer bagian adilnya hanya 1,56%, sehingga terberat **4,28%** (hampir 3 kali lipat bagian adil). Rasio terhadap bagian adil justru **memburuk** saat reducer ditambah: 1,17× (4) → 1,41× (16) → 2,74× (64). Ini **skew yang berasal dari data (satu key), bukan dari jumlah mesin**.
- Dengan combiner, setiap kata hanya dikirim **sekali per mapper**, jadi key `di` hanya berupa **9 pasangan** (9 mapper) dari 74.009 pasangan (± 0,012%). Tidak ada lagi key yang mendominasi, sehingga beban terdistribusi hampir rata (pada 4 reducer: 24,7%–25,6%), dan rasio terhadap bagian adil jauh lebih dekat ke 1× (1,02× pada 4 reducer, 1,05× pada 16, 1,19× pada 64, dibanding 1,17×, 1,41×, 2,74× tanpa combiner).

## L-3. Haruskah `rp`, `ri`, dan `the` ditambahkan ke stopword?
**Pertanyaan:** Kata "rp", "ri", dan "the" muncul di antara kata teratas setelah stopword dibuang. Apakah sebaiknya ditambahkan ke daftar stopword? Jelaskan dari sisi tujuan analisis.

### Kode: lihat sebaran dan konteks ketiga kata, lalu bandingkan top-10 sebelum/sesudah

In [28]:
sorotan = ["rp", "ri", "the"]
peringkat = {k: i + 1 for i, (k, n) in enumerate(top)}

# (a) Sebaran per kategori (hasil H-13) dan peringkat global (tanpa stopword)
tabel = pd.DataFrame({kata: {kat: gabung[(kat, kata)] for kat in sorted(per_kategori)} for kata in sorotan})
tabel.loc["TOTAL"] = tabel.sum()
print(tabel.to_string())
print("\nPeringkat global tanpa stopword:", {k: peringkat[k] for k in sorotan})

# (b) Konteks: contoh judul yang memuat kata tersebut
for kata in sorotan:
    contoh_j = bersih.loc[bersih["judul"].str.lower().str.contains(rf"\b{kata}\b", regex=True), ["kategori", "judul"]].head(3)
    print(f"\nContoh judul dengan '{kata}':")
    for _, r_ in contoh_j.iterrows():
        print(f"  [{r_['kategori']}] {r_['judul']}")

# (c) Top-10 sebelum vs sesudah ketiga kata ditambahkan ke stopword
sesudah = [k for k, n in top if k not in set(sorotan)][:10]
print("\nTop-10 sebelum:", [k for k, n in top[:10]])
print("Top-10 sesudah:", sesudah)

           rp    ri  the
finance  1385  1125    8
food      100     3   24
health     26   196    4
hot       315     6  528
inet      210    19   16
news      778   680   50
oto       267   126    7
sport       9     8   17
travel     64    43   24
TOTAL    3154  2206  678

Peringkat global tanpa stopword: {'rp': 2, 'ri': 12, 'the': 102}

Contoh judul dengan 'rp':
  [finance] BRI Gelar Bazar Ragam Merek Sneaker, Ada Cashback hingga Rp 2 Juta
  [finance] Ada Anggaran Rp 72 M untuk Influencer, Dipakai Buat Apa Sih?
  [finance] Wishnutama Mau Gelontorkan Rp 72 M untuk Influencer Asing, Buat Apa?

Contoh judul dengan 'ri':
  [finance] RI Bakal Punya Destinasi Wisata Sejarah Islam di Ancol
  [finance] CT Sebut Virus Baru Masuk RI, Namanya Jiwasraya
  [finance] Aturan Wajib Penggunaan Kapal RI Bisa Ganggu Ekspor Batu Bara?

Contoh judul dengan 'the':
  [finance] Erick Thohir Beberkan Rencana Bisnis 'The New Oil' Telkom
  [finance] Bahas Proyeksi Perikanan Budi Daya, Edhy Prabowo: Harus Out 

**Interpretasi:** keputusan stopword bergantung pada **tujuan analisis**. Tujuan kita adalah mengetahui **topik apa yang dibahas** dan **apakah topik berbeda antar-kategori**.
- **`the`** (678 kemunculan, peringkat global ke-102, tetapi masuk top-5 kategori `hot` dengan 528): bukan kata Indonesia, melainkan potongan frasa/nama berbahasa Inggris (contoh: "The Fed", "The New Oil", "Out of The Box"). Ia tidak menunjuk topik apa pun, jadi **sebaiknya ditambahkan** ke stopword.
- **`ri`** (2.206 kemunculan, peringkat ke-12): singkatan "Republik Indonesia" yang dipakai sebagai sebutan "Indonesia/negara" pada banyak topik ("RI Bakal Punya...", "Virus Baru Masuk RI"). Ia label umum, bukan topik itu sendiri; untuk analisis topik **sebaiknya ditambahkan**.
- **`rp`** (3.154 kemunculan, peringkat ke-2 setelah corona): singkatan "rupiah" yang menandai adanya nominal uang ("Rp 72 M", "Rp 2 Juta"). Kata ini sangat dominan di `finance` (1.385) dan `news` (778) tetapi tidak menyebut topik tertentu. Untuk tujuan "topik apa yang dibahas" sebaiknya ditambahkan, **dengan catatan di laporan** bahwa judul ekonomi memang sarat angka rupiah.
- Dampaknya terlihat pada top-10: setelah `rp` dibuang, kata `minta` masuk ke peringkat 10 dan urutan lainnya bergeser naik, jadi bahasan substantif (virus, kasus, polisi, ...) tidak lagi terdesak kata yang tidak membedakan kategori. Tetapi stopword **bukan daftar tetap**: bila tujuan analisisnya berbeda (mis. mengukur seberapa sering berita menyebut nominal uang), `rp` justru harus dipertahankan. Apapun keputusannya, alasannya dicatat di laporan.

## L-4. Dampak url rusak pada kategori `inet`
**Pertanyaan:** Seluruh baris kategori `inet` punya url rusak. Apa dampaknya bila url dipakai sebagai kunci join pada pipeline lanjutan? Mengapa baris itu tidak dibuang?

### Kode: tunjukkan url rusak, simulasikan join, dan hitung yang hilang bila dibuang

In [29]:
# (a) Kondisi url pada data mentah
inet_mentah = mentah[mentah["kategori"] == "inet"]
print("Baris inet (mentah) :", f"{len(inet_mentah):,}")
print("Nilai url yang ada  :", inet_mentah["url"].unique().tolist(), "| jumlah unik:", inet_mentah["url"].nunique())
print("Kategori lain, url unik:", mentah.loc[mentah["kategori"] != "inet", "url"].nunique(),
      "dari", f"{(mentah['kategori'] != 'inet').sum():,}", "baris")

# (b) Simulasi join pada url (200 baris contoh): inet vs kategori lain
a = inet_mentah.head(200)
b_ = mentah[mentah["kategori"] == "finance"].head(200)
print("\nSelf-join inet (200 baris) pada url   ->", f"{len(a.merge(a, on='url')):,}", "baris hasil (harusnya 200)")
print("Self-join finance (200 baris) pada url ->", f"{len(b_.merge(b_, on='url')):,}", "baris hasil")
n = len(inet_mentah)
print(f"Self-join seluruh inet pada url '#' -> {n:,} x {n:,} = {n*n:,} baris (ledakan many-to-many)")

# (c) Setelah dibersihkan: url dan id dikosongkan (NaN) sehingga tidak dipakai sebagai kunci
inet_bersih = bersih[bersih["kategori"] == "inet"]
print("\nSetelah cleaning -> baris inet:", f"{len(inet_bersih):,}",
      "| url NaN:", int(inet_bersih["url"].isna().sum()), "| id NaN:", int(inet_bersih["id"].isna().sum()))

# (d) Kerugian bila baris inet dibuang
print(f"\nBila dibuang: hilang {len(inet_bersih)/len(bersih)*100:.2f}% judul dan satu kategori utuh dari 9.")
print("Kata teratas inet (hasil H-13):",
      ", ".join(f"{k} ({n_:,})" for n_, k in sorted(per_kategori["inet"], reverse=True)[:5]))

Baris inet (mentah) : 5,640
Nilai url yang ada  : ['#'] | jumlah unik: 1
Kategori lain, url unik: 85377 dari 85,377 baris

Self-join inet (200 baris) pada url   -> 40,000 baris hasil (harusnya 200)
Self-join finance (200 baris) pada url -> 200 baris hasil
Self-join seluruh inet pada url '#' -> 5,640 x 5,640 = 31,809,600 baris (ledakan many-to-many)

Setelah cleaning -> baris inet: 5,577 | url NaN: 5577 | id NaN: 5577

Bila dibuang: hilang 6.14% judul dan satu kategori utuh dari 9.
Kata teratas inet (hasil H-13): corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)


**Interpretasi:**
- **Dampak pada join:** semua baris `inet` punya `url = '#'`, jadi nilai kuncinya **identik** untuk 5.640 baris. Join pada url menggabungkan *setiap baris dengan setiap baris lain* (hubungan many-to-many): 5.640 × 5.640 ≈ 31,8 juta pasangan palsu, dan pada simulasi 200 baris saja hasil join melonjak dari 200 menjadi 40.000. Hasilnya angka menggelembung, duplikasi, dan korelasi semu. Selain itu `#` bukan alamat artikel sehingga tidak bisa dipakai menautkan ke sumber.
- **Perbaikan di pipeline ini:** kolom `url` dikosongkan (`NaN`) dan `id` dibiarkan kosong, sehingga baris `inet` tidak ikut tergabung secara keliru (join pada `NaN` tidak mencocokkan baris).
- **Mengapa baris tidak dibuang:** yang rusak hanya kolom `url`; **judul, kategori, dan tanggalnya sah**. Menghapusnya berarti membuang **satu kategori utuh (± 6% judul)** dari analisis, padahal pertanyaan kita justru membandingkan topik antar-kategori. Judul `inet` masih berguna untuk analisis teks (kata teratas: corona, virus, ponsel, ...). Prinsipnya: pilihan membuang / memperbaiki / mempertahankan baris cacat adalah **keputusan analisis** dan alasannya dicatat di laporan.

## L-5. Mengapa partisi `kategori` dipilih dibanding `kategori + bulan`?
**Pertanyaan:** Mengapa partisi kategori dipilih dibanding kategori + bulan untuk MapReduce di H-9? Hubungkan dengan jumlah mapper dan small file problem.

### Kode: bandingkan jumlah mapper, ukuran rata-rata split, dan waktu baca

In [30]:
def ringkas_partisi(kolom, folder="uji_partisi_l5"):
    n, mb = simpan_terpartisi(bersih, kolom, folder)
    berkas = sorted(Path(folder).rglob("*.parquet"))
    baris_per_berkas = [len(pd.read_parquet(p, columns=["judul"])) for p in berkas]
    t0 = time.perf_counter()
    for p in berkas:                       # biaya "membuka" semua split satu per satu
        pd.read_parquet(p, columns=["judul"])
    waktu_baca = time.perf_counter() - t0
    kb_rata = sum(p.stat().st_size for p in berkas) / len(berkas) / 1e3
    shutil.rmtree(folder)
    return {"mapper (= jumlah file)": n, "total MB": round(mb, 1), "KB rata-rata/file": round(kb_rata, 1),
            "judul min/file": min(baris_per_berkas), "judul maks/file": max(baris_per_berkas),
            "waktu buka semua file (s)": round(waktu_baca, 3)}

ringkasan_partisi = {str(k): ringkas_partisi(k) for k in (["kategori"], ["bulan"], ["kategori", "bulan"])}
print(pd.DataFrame.from_dict(ringkasan_partisi, orient="index").to_string())

                       mapper (= jumlah file)  total MB  KB rata-rata/file  judul min/file  judul maks/file  waktu buka semua file (s)
['kategori']                                9       8.4              929.4            2433            32321                      0.058
['bulan']                                   6       8.3             1384.0            7334            18103                      0.058
['kategori', 'bulan']                      54       8.4              156.4             190             6633                      0.121


**Interpretasi:**
- Pada MapReduce, **satu berkas partisi = satu input split = satu mapper**. Partisi `kategori` menghasilkan **9 mapper**; `kategori + bulan` menghasilkan **54 mapper** untuk volume data yang hampir sama (± 8,4 MB), sehingga rata-rata ukuran berkas turun dari ± 930 KB menjadi ± 156 KB (berkas terkecil hanya 190 judul).
- Itulah **small file problem**: banyak berkas kecil berarti banyak tugas kecil. Setiap mapper punya biaya tetap (menyalakan tugas, membuka berkas, membaca metadata Parquet) yang tidak sebanding dengan pekerjaannya; pada cluster Hadoop, banyak berkas kecil juga membebani NameNode (metadata) dan penjadwalan YARN. Pada tabel, waktu membuka seluruh berkas ikut naik pada 54 berkas (± 3× lebih lama pada percobaan ini, padahal datanya sama; angka waktu bervariasi antar-komputer).
- Partisi `bulan` hanya 6 berkas, tetapi pertanyaan kita menyangkut **kategori**, sehingga partisi kategori sekaligus mempercepat query per kategori (*partition pruning*, hanya 1 dari 9 berkas dibaca). Jadi `kategori` adalah titik tengah: cukup sedikit untuk menghindari small file problem, cukup banyak untuk paralelisme (walau ukuran split tidak seimbang: `news` 32 ribu vs `sport` 2 ribu judul).

## L-6. Mengapa hasil harus diurutkan sebelum masuk ke `reducer.py`?
**Pertanyaan:** Pada H-12, mengapa hasil harus diurutkan sebelum masuk ke reducer.py? Apa yang terjadi pada hasil bila sort dihilangkan?

### Kode: jalankan pipeline tanpa `sort` dan bandingkan dengan hasil yang benar

In [31]:
import subprocess

def jalankan_pipe(pakai_sort):
    perintah = ("cat masukan.tsv | python3 mapper.py | "
                + ("LC_ALL=C sort | " if pakai_sort else "") + "python3 reducer.py")
    out = subprocess.run(perintah, shell=True, capture_output=True, text=True, executable="/bin/bash").stdout
    return [b.split("\t") for b in out.splitlines()]

tanpa_sort = jalankan_pipe(pakai_sort=False)
print("Baris keluaran dengan sort (H-12)   :", f"{len(hasil_pipe):,}", "(= jumlah kata unik)")
print("Baris keluaran tanpa sort           :", f"{len(tanpa_sort):,}")

# Berapa baris yang dikeluarkan untuk kata 'corona'?
baris_corona = [int(n) for k, n in tanpa_sort if k == "corona"]
print(f"\n'corona' tanpa sort muncul di {len(baris_corona):,} baris, contoh 5 nilai pertama: {baris_corona[:5]}")
print(f"Jumlah semua barisnya = {sum(baris_corona):,} (benar: {hasil_semua['corona']:,}), "
      f"tetapi tidak ada satu baris pun yang bernilai {hasil_semua['corona']:,}")

# Bila dibaca sebagai kamus (kunci unik), hasilnya salah
salah = {}
for k, n in tanpa_sort:
    salah[k] = int(n)
print("\nHasil tanpa sort sama dengan H-9?", salah == hasil_semua)
print("Kata dengan nilai salah:", f"{sum(1 for k in hasil_semua if salah.get(k) != hasil_semua[k]):,}", "dari", f"{len(hasil_semua):,}")

Baris keluaran dengan sort (H-12)   : 32,275 (= jumlah kata unik)
Baris keluaran tanpa sort           : 821,949

'corona' tanpa sort muncul di 17,210 baris, contoh 5 nilai pertama: [1, 1, 1, 1, 1]
Jumlah semua barisnya = 17,239 (benar: 17,239), tetapi tidak ada satu baris pun yang bernilai 17,239

Hasil tanpa sort sama dengan H-9? False
Kata dengan nilai salah: 21,214 dari 32,275


**Interpretasi:**
- `reducer.py` hanya menyimpan **satu key dan satu total** pada satu waktu: selama key sama ia menambahkan, begitu key **berganti** ia mencetak hasil key sebelumnya dan mulai dari nol. Logika ini hanya benar bila **semua baris dengan key yang sama berdampingan**, yaitu setelah diurutkan. Di Hadoop, urutan ini dijamin oleh tahap **shuffle & sort**; pada pipe Linux perannya dipegang `sort`.
- Tanpa `sort`, baris keluaran mapper masih berurutan **menurut judul**, bukan menurut kata. Setiap kali kata berganti (hampir di setiap baris), reducer menutup key lama dan mencetaknya, sehingga satu kata yang sama dicetak **berkali-kali** dengan hitungan parsial. Output tanpa sort membengkak menjadi ± 822 ribu baris (seharusnya 32.275), kata `corona` muncul di ± 17 ribu baris dengan nilai kecil (hampir semuanya 1), dan **tidak ada baris yang memuat total sebenarnya (17.239)**. Walaupun jumlah seluruh barisnya masih 17.239, saat dibaca sebagai kamus sekitar 21 ribu dari 32.275 kata bernilai salah (hasil ≠ H-9).
- Jadi `sort` bukan sekadar merapikan: ia **mengelompokkan key yang sama**, dan itulah inti shuffle.

## L-7. Mengapa H-4 memakai sitemap berita, bukan membuka halaman setiap artikel?
**Pertanyaan:** Jelaskan untung ruginya bagi server detik dan bagi kode Anda.

### Kode: hitung beban request, dan uji ketahanan parser terhadap perubahan tampilan

In [32]:
import math

# (a) Beban request untuk 20 judul (nilai dari konstanta H-4) dan untuk seluruh dataset
n_host = len({urlparse(s).netloc for s in SITEMAP_DIPILIH})
req_sitemap = len(SITEMAP_DIPILIH) + n_host      # 3 sitemap + robots.txt tiap host
req_artikel = BATAS_TOTAL + n_host               # 20 artikel + robots.txt tiap host
print(f"{'cara':<26}{'request konten':>16}{'+ robots.txt':>14}{'waktu min. (s)':>16}")
print(f"{'sitemap berita':<26}{len(SITEMAP_DIPILIH):>16}{n_host:>14}{len(SITEMAP_DIPILIH)*DELAY:>16.0f}")
print(f"{'buka tiap artikel':<26}{BATAS_TOTAL:>16}{n_host:>14}{BATAS_TOTAL*DELAY:>16.0f}")

JUDUL_PER_SITEMAP = 200   # satu sitemap berita berisi 200 entri (uji dosen)
print(f"\nUntuk 91.017 judul: sitemap = {math.ceil(91017/JUDUL_PER_SITEMAP):,} request, "
      f"buka artikel = {91017:,} request  ({91017/math.ceil(91017/JUDUL_PER_SITEMAP):.0f}x lebih banyak)")

# (b) Data apa saja yang didapat dari satu berkas sitemap, tanpa membuka halaman artikel
contoh_1 = parse_sitemap_news(CONTOH_XML)[0]
print("\nDari sitemap saja:", {k: contoh_1[k] for k in ("judul", "tanggal", "kata_kunci")})

# (c) Ketahanan: HTML portal diubah tampilannya (nama class berganti) -> parser HTML rusak
html_baru = ('<article class="item"><h2 class="headline"><a href="/x">Judul</a></h2>'
             '<span class="kategori">news</span><time datetime="2020-01-01">01/01/2020</time></article>')
try:
    print("\nParser HTML pada tampilan baru:", parse_halaman(html_baru))
except Exception as e:
    print("\nParser HTML pada tampilan baru GAGAL ->", type(e).__name__, ":", e)
print("Parser sitemap pada format XML yang sama tetap bekerja:", len(parse_sitemap_news(CONTOH_XML)), "entri")

cara                        request konten  + robots.txt  waktu min. (s)
sitemap berita                           3             3               6
buka tiap artikel                       20             3              40

Untuk 91.017 judul: sitemap = 456 request, buka artikel = 91,017 request  (200x lebih banyak)

Dari sitemap saja: {'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'tanggal': '2026-10-29', 'kata_kunci': 'mahkamah konstitusi,mk,hukum,kepala daerah'}

Parser HTML pada tampilan baru GAGAL -> AttributeError : 'NoneType' object has no attribute 'get_text'
Parser sitemap pada format XML yang sama tetap bekerja: 3 entri


**Interpretasi:**
- **Untung bagi server detik:** request jauh lebih sedikit. Untuk 20 judul hanya **3 request konten** (vs 20 bila membuka artikel); untuk skala 91.017 judul selisihnya ratusan kali lipat (± 456 vs 91.017 request). Server tidak perlu melayani render halaman penuh, gambar, iklan, dan skrip, jadi beban lebih ringan dan crawling lebih sopan.
- **Untung bagi kode kita:** sitemap berita sudah memuat **judul, tanggal terbit, dan kata kunci** dalam format XML standar (sitemaps.org), jadi satu berkas XML memberi puluhan judul sekaligus. Parser tidak ikut rusak ketika **tampilan HTML berubah**, seperti terbukti pada uji (c): parser HTML gagal ketika nama class berganti, sedangkan parser sitemap tetap bekerja. Kode juga lebih singkat dan lebih cepat.
- **Rugi / keterbatasan:** (1) hanya berisi **judul**, tanpa isi artikel, jadi tidak cocok bila butuh teks lengkap; (2) satu sitemap hanya memuat entri **terbaru** (± 200 entri), jadi tidak bisa mengambil arsip lama; (3) bergantung pada situs menyediakan sitemap yang akurat, dan pada pengamatan dosen tanda zona waktunya tampak keliru sehingga hanya tanggal yang dipakai; (4) hak cipta tetap milik detik: "ketiadaan larangan bukan izin", jadi data hanya dipakai untuk pembelajaran dan tidak dibagikan ulang.

---
# M. Studi Kasus
## Apa yang dibahas portal berita pada paruh pertama 2020?
Pimpinan redaksi meminta ringkasan satu halaman berdasarkan data judul. Temuan dari H-7 dan H-13 dipakai.

### Kode: puncak topik corona per bulan

In [33]:
corona_bulan = con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df()
total_bulan = con.sql("SELECT bulan, COUNT(*) AS semua_judul FROM berita GROUP BY bulan ORDER BY bulan").df()
tabel_bulan = corona_bulan.merge(total_bulan, on="bulan")
tabel_bulan["persen_corona"] = (tabel_bulan["judul_corona"] / tabel_bulan["semua_judul"] * 100).round(1)
tabel_bulan.loc[tabel_bulan["bulan"] == "2020-06", "bulan"] = "2020-06 (13 hari)"
print(tabel_bulan.to_string(index=False))
puncak = tabel_bulan.loc[tabel_bulan["judul_corona"].idxmax()]
print(f"\nPuncak (jumlah judul)  : {puncak['bulan']} = {puncak['judul_corona']:,} judul")
print(f"Puncak (persen judul)  : {tabel_bulan.loc[tabel_bulan['persen_corona'].idxmax(), 'bulan']} = {tabel_bulan['persen_corona'].max()}% dari seluruh judul bulan itu")

            bulan  judul_corona  semua_judul  persen_corona
          2020-01           385        16258            2.4
          2020-02          1077        16135            6.7
          2020-03          6994        18103           38.6
          2020-04          4913        16636           29.5
          2020-05          2934        16352           17.9
2020-06 (13 hari)           943         7334           12.9

Puncak (jumlah judul)  : 2020-03 = 6,994 judul
Puncak (persen judul)  : 2020-03 = 38.6% dari seluruh judul bulan itu


**Interpretasi:** jumlah judul tentang corona naik dari 385 (Januari) menjadi 1.077 (Februari), **memuncak pada Maret 2020 (6.994 judul)**, lalu menurun ke 4.913 (April) dan 2.934 (Mei). Juni (943 judul) hanya mencakup 13 hari pertama sehingga tidak boleh dibandingkan langsung dengan bulan penuh; itulah mengapa kolom persen terhadap seluruh judul bulan itu ikut ditampilkan.

### Kode: kategori yang paling banyak membahas dan yang paling berbeda

In [34]:
# (a) Kategori yang paling banyak membahas corona (jumlah dan proporsi judul)
corona_kat = con.sql("""
    SELECT kategori, COUNT(*) AS judul,
           SUM(CASE WHEN lower(judul) LIKE '%corona%' THEN 1 ELSE 0 END) AS judul_corona
    FROM berita GROUP BY kategori""").df()
corona_kat["judul_corona"] = corona_kat["judul_corona"].astype(int)
corona_kat["persen_corona"] = (corona_kat["judul_corona"] / corona_kat["judul"] * 100).round(1)
print(corona_kat.sort_values("persen_corona", ascending=False).to_string(index=False))

# (b) Kategori yang bahasannya paling berbeda: kemiripan 20 kata teratas (Jaccard) dengan kategori lain
ABAIKAN = {"rp", "ri", "the", "foto", "galeri"}      # singkatan/penanda format (lihat L-3 dan Latihan 1)
top20 = {kat: set([k for n_, k in sorted(per_kategori[kat], reverse=True) if k not in ABAIKAN][:20])
         for kat in per_kategori}
rata_jaccard = {}
for kat in top20:
    skor = [len(top20[kat] & top20[o]) / len(top20[kat] | top20[o]) for o in top20 if o != kat]
    rata_jaccard[kat] = sum(skor) / len(skor)
print("\nKemiripan rata-rata 20 kata teratas dengan kategori lain (makin kecil = makin berbeda):")
for kat, v in sorted(rata_jaccard.items(), key=lambda x: x[1]):
    lima = [k for n_, k in sorted(per_kategori[kat], reverse=True) if k not in ABAIKAN][:5]
    print(f"  {kat:<8} {v:.3f}   5 kata teratas: {', '.join(lima)}")

kategori  judul  judul_corona  persen_corona
  health   4914          2591           52.7
    news  32321          7532           23.3
 finance  14141          2473           17.5
  travel   6453          1025           15.9
    inet   5577           847           15.2
     oto   4375           577           13.2
     hot  16290          1663           10.2
   sport   2433           228            9.4
    food   4314           310            7.2

Kemiripan rata-rata 20 kata teratas dengan kategori lain (makin kecil = makin berbeda):
  sport    0.057   5 kata teratas: corona, indonesia, motogp, masters, olimpiade
  finance  0.057   5 kata teratas: corona, harga, jokowi, mau, ekonomi
  travel   0.082   5 kata teratas: corona, wisata, dunia, turis, indonesia
  news     0.083   5 kata teratas: corona, polisi, kasus, warga, positif
  food     0.090   5 kata teratas: makanan, makan, enak, bikin, rumah
  hot      0.094   5 kata teratas: corona, anak, rumah, meninggal, film
  oto      0.128   

**Interpretasi:**
- **Kategori yang paling banyak membahas corona:** secara **proporsi**, `health` paling intens: **52,7%** judulnya (2.591 dari 4.914) memuat corona. Secara **jumlah**, `news` terbanyak: **7.532 judul (23,3%)**. Disusul `finance` (17,5%), `travel` (15,9%), dan `inet` (15,2%).
- **Kategori yang bahasannya paling berbeda:** `food` hanya **7,2%** judulnya memuat corona (terendah), dan kata teratasnya khas kuliner: makanan, makan, enak, resep, masak. `sport` (9,4%) dan `finance` memiliki kosakata yang paling sedikit beririsan dengan kategori lain (kemiripan rata-rata 0,057, terendah): sport dengan motogp, masters, olimpiade, rossi, marquez; finance dengan harga, ekonomi, jokowi. Sebaliknya `health` dan `inet` paling mirip dengan kategori lain karena sama-sama didominasi corona dan virus.
- Artinya topik **berbeda antar-kategori**, tetapi corona menjadi "tema bersama" yang menembus hampir semua kategori (kecuali food yang lebih tahan).

### Ringkasan untuk pimpinan redaksi (150–200 kata)

In [35]:
ringkasan = """Berdasarkan 90.818 judul detik.com periode 1 Januari–13 Juni 2020, topik yang paling dominan adalah pandemi corona. Jumlah judul yang memuat kata corona naik dari 385 pada Januari menjadi 1.077 pada Februari, lalu memuncak pada Maret 2020 dengan 6.994 judul (38,6% dari seluruh judul bulan itu), kemudian turun menjadi 4.913 pada April dan 2.934 pada Mei. Kategori health paling intens membahasnya: 52,7% judulnya memuat corona. Secara jumlah, kategori news terbanyak dengan 7.532 judul. Bahasan yang paling berbeda ada pada food dan sport. Di food hanya 7,2% judul menyinggung corona, dengan kata teratas makanan, makan, dan enak. Sport, bersama finance, memiliki kosakata yang paling sedikit beririsan dengan kategori lain, dengan kata khas seperti motogp, masters, dan olimpiade. Keterbatasan data: pertama, hanya judul yang tersedia, bukan isi artikel, sehingga kedalaman liputan tidak dapat dinilai. Kedua, Juni hanya mencakup 13 hari sehingga tidak sebanding dengan bulan penuh. Ketiga, seluruh url kategori inet rusak dan dataset dikumpulkan pihak ketiga tanpa lisensi, sehingga perlu diperiksa sebelum dipublikasikan."""
print(ringkasan)
print("\nJumlah kata:", len(ringkasan.split()))

Berdasarkan 90.818 judul detik.com periode 1 Januari–13 Juni 2020, topik yang paling dominan adalah pandemi corona. Jumlah judul yang memuat kata corona naik dari 385 pada Januari menjadi 1.077 pada Februari, lalu memuncak pada Maret 2020 dengan 6.994 judul (38,6% dari seluruh judul bulan itu), kemudian turun menjadi 4.913 pada April dan 2.934 pada Mei. Kategori health paling intens membahasnya: 52,7% judulnya memuat corona. Secara jumlah, kategori news terbanyak dengan 7.532 judul. Bahasan yang paling berbeda ada pada food dan sport. Di food hanya 7,2% judul menyinggung corona, dengan kata teratas makanan, makan, dan enak. Sport, bersama finance, memiliki kosakata yang paling sedikit beririsan dengan kategori lain, dengan kata khas seperti motogp, masters, dan olimpiade. Keterbatasan data: pertama, hanya judul yang tersedia, bukan isi artikel, sehingga kedalaman liputan tidak dapat dinilai. Kedua, Juni hanya mencakup 13 hari sehingga tidak sebanding dengan bulan penuh. Ketiga, seluruh

**Interpretasi:** ringkasan memuat tiga hal yang diminta: (a) kapan topik memuncak, (b) kategori terbanyak dan paling berbeda, (c) keterbatasan data. Semua angkanya berasal dari output kode di atas (H-7 dan H-13), dan jumlah kata dihitung otomatis agar berada pada rentang 150–200.

---
# N. Latihan

## Latihan 1. Stopword domain
**Tugas:** tambahkan 5 kata ke `STOPWORD` yang mengganggu hasil, jalankan ulang H-13, catat kata yang masuk dan perubahan lima kata teratas tiap kategori.

### Kode: lima kata tambahan dan perbandingan top-5 sebelum/sesudah

In [36]:
KATA_TAMBAHAN = {"rp", "ri", "the", "foto", "galeri"}
STOPWORD_BARU = STOPWORD | KATA_TAMBAHAN

def mapper_kategori_sw(path, stop):
    """Sama dengan mapper_kategori (H-13), tetapi daftar stopword menjadi parameter."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in stop:
                c[(kategori, kata)] += 1
    return c

gabung_baru = collections.Counter()
for c in map(lambda p: mapper_kategori_sw(p, STOPWORD_BARU), splits):
    gabung_baru.update(c)

def top5_per_kategori(g):
    tmp = collections.defaultdict(list)
    for (kat, kata), n in g.items():
        tmp[kat].append((n, kata))
    return {kat: [k for n, k in sorted(v, reverse=True)[:5]] for kat, v in tmp.items()}

sebelum, sesudah = top5_per_kategori(gabung), top5_per_kategori(gabung_baru)
print("Kata yang ditambahkan ke STOPWORD:", sorted(KATA_TAMBAHAN))
print(f"\n{'kategori':<9}{'sebelum':<46}{'sesudah':<46}")
for kat in sorted(sebelum):
    print(f"{kat:<9}{', '.join(sebelum[kat]):<46}{', '.join(sesudah[kat]):<46}")

print("\nPerubahan per kategori:")
for kat in sorted(sebelum):
    keluar = [k for k in sebelum[kat] if k not in sesudah[kat]]
    masuk = [k for k in sesudah[kat] if k not in sebelum[kat]]
    if keluar or masuk:
        print(f"  {kat:<8} keluar: {keluar}  masuk: {masuk}")

Kata yang ditambahkan ke STOPWORD: ['foto', 'galeri', 'ri', 'rp', 'the']

kategori sebelum                                       sesudah                                       
finance  corona, rp, ri, harga, jokowi                 corona, harga, jokowi, mau, ekonomi           
food     makanan, makan, foto, galeri, enak            makanan, makan, enak, bikin, rumah            
health   corona, virus, pasien, indonesia, meninggal   corona, virus, pasien, indonesia, meninggal   
hot      corona, anak, rumah, meninggal, the           corona, anak, rumah, meninggal, film          
inet     corona, virus, ponsel, bikin, indonesia       corona, virus, ponsel, bikin, indonesia       
news     corona, polisi, kasus, warga, positif         corona, polisi, kasus, warga, positif         
oto      mobil, corona, motor, rp, indonesia           mobil, corona, motor, indonesia, honda        
sport    corona, indonesia, motogp, masters, olimpiade corona, indonesia, motogp, masters, olimpiade 
travel  

**Interpretasi:** lima kata yang dipilih adalah `rp`, `ri`, `the` (singkatan/kata asing yang tidak menunjuk topik; ketiganya muncul pada top-5 `finance`, `oto`, `hot`) serta `foto` dan `galeri` (kata penanda format konten "foto/galeri" pada `food` dan `travel`, bukan topik berita). Setelah ditambahkan, kata-kata itu **keluar** dari top-5 dan digantikan kata di peringkat berikutnya: `finance` (keluar rp, ri; masuk mau, ekonomi), `food` (keluar foto, galeri; masuk bikin, rumah), `hot` (keluar the; masuk film), `oto` (keluar rp; masuk honda), `travel` (keluar foto; masuk indonesia). Kategori yang tidak memuat kata-kata itu di top-5 (`health`, `news`, `sport`, `inet`) tidak berubah. Pelajaran: stopword domain adalah **keputusan analisis** dan perlu dicatat di laporan.

## Latihan 2. Per bulan
**Tugas:** ubah key di H-13 menjadi `(bulan, kata)`. Kata apa yang berubah dari Januari ke Mei?

### Kode: mapper dengan key `(bulan, kata)`

In [37]:
def mapper_bulan(path):
    """Key gabungan (bulan, kata); combiner Counter di dalam mapper."""
    df_ = pd.read_parquet(path, columns=["judul", "bulan"])
    c = collections.Counter()
    for t, b in zip(df_["judul"], df_["bulan"]):
        for kata in tokenisasi(t):
            if kata not in STOPWORD_BARU:
                c[(b, kata)] += 1
    return c

gabung_bulan = collections.Counter()
for c in map(mapper_bulan, splits):
    gabung_bulan.update(c)

per_bulan = collections.defaultdict(list)
for (b, kata), n in gabung_bulan.items():
    per_bulan[b].append((n, kata))
top_bulan = {b: sorted(v, reverse=True)[:10] for b, v in per_bulan.items()}

for b in sorted(top_bulan):
    print(f"{b}: ", ", ".join(f"{k} ({n:,})" for n, k in top_bulan[b][:6]))

jan, mei = [k for n, k in top_bulan["2020-01"]], [k for n, k in top_bulan["2020-05"]]
print("\nTop-10 Januari :", jan)
print("Top-10 Mei     :", mei)
print("Hanya di Januari:", [k for k in jan if k not in mei])
print("Hanya di Mei    :", [k for k in mei if k not in jan])

2020-01:  banjir (1,220), polisi (554), jokowi (516), tahun (493), baru (475), mobil (428)
2020-02:  corona (1,076), polisi (630), virus (596), banjir (572), anak (453), indonesia (415)
2020-03:  corona (6,986), virus (1,340), positif (925), pasien (749), rumah (695), cegah (665)
2020-04:  corona (4,914), positif (828), pandemi (706), psbb (689), warga (609), meninggal (577)
2020-05:  corona (2,935), kasus (640), positif (628), pandemi (596), warga (565), lebaran (562)
2020-06:  corona (943), normal (471), new (465), kasus (315), juni (262), polisi (246)

Top-10 Januari : ['banjir', 'polisi', 'jokowi', 'tahun', 'baru', 'mobil', 'minta', 'kpk', 'indonesia', 'china']
Top-10 Mei     : ['corona', 'kasus', 'positif', 'pandemi', 'warga', 'lebaran', 'psbb', 'covid', 'polisi', 'normal']
Hanya di Januari: ['banjir', 'jokowi', 'tahun', 'baru', 'mobil', 'minta', 'kpk', 'indonesia', 'china']
Hanya di Mei    : ['corona', 'kasus', 'positif', 'pandemi', 'warga', 'lebaran', 'psbb', 'covid', 'normal']


**Interpretasi:** dengan key `(bulan, kata)` setiap mapper tetap memproses satu partisi, tetapi tiap kata dihitung per bulan (stopword yang dipakai: `STOPWORD_BARU` dari Latihan 1). **Januari** didominasi `banjir` (1.220), `polisi`, `jokowi`, `tahun`, `baru`, `mobil`, `kpk`, dan `china`; `corona` belum masuk sepuluh besar. Pada **Mei** daftar teratas berubah menjadi `corona`, `kasus`, `positif`, `pandemi`, `warga`, `lebaran`, `psbb`, `covid`, `normal`. Kata yang **hanya ada di Januari**: banjir, jokowi, tahun, baru, mobil, minta, kpk, indonesia, china; **hanya di Mei**: corona, kasus, positif, pandemi, warga, lebaran, psbb, covid, normal. Jadi agenda berita bergeser dari banjir/politik/otomotif menuju pandemi (kasus positif, PSBB, new normal). Juni hanya 13 hari sehingga tidak dibandingkan.

## Latihan 3. Bigram
**Tugas:** ubah mapper agar menghasilkan pasangan kata berurutan, misalnya "virus corona". Berapa pasangan yang dihasilkan, dan berapa setelah combiner?

### Kode: mapper bigram tanpa dan dengan combiner

In [38]:
def mapper_bigram(path):
    """Tanpa combiner: satu pasangan (bigram, 1) per bigram."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    hasil_ = []
    for t in judul:
        tok = tokenisasi(t)
        hasil_ += [(f"{a} {b}", 1) for a, b in zip(tok, tok[1:])]
    return hasil_

def mapper_bigram_combiner(path):
    c = collections.Counter()
    for bg, n in mapper_bigram(path):
        c[bg] += n
    return c

bg_map = [mapper_bigram(p) for p in splits]
bg_comb = [mapper_bigram_combiner(p) for p in splits]
n_tanpa, n_dengan = sum(len(x) for x in bg_map), sum(len(x) for x in bg_comb)
print(f"Pasangan bigram tanpa combiner : {n_tanpa:,}")
print(f"Pasangan bigram dengan combiner: {n_dengan:,}  (turun {n_tanpa/n_dengan:.1f}x)")

total_bg = collections.Counter()
for c in bg_comb:
    total_bg.update(c)
print("Bigram unik:", f"{len(total_bg):,}")
print("\n10 bigram teratas (semua):", total_bg.most_common(10))
substantif = [(b, n) for b, n in total_bg.most_common() if not any(w in STOPWORD_BARU for w in b.split())]
print("\n10 bigram teratas (tanpa stopword di kedua kata):")
for b, n in substantif[:10]:
    print(f"  {b:<22}{n:>7,}")
print("\n'virus corona':", f"{total_bg['virus corona']:,}")

Pasangan bigram tanpa combiner : 734,783
Pasangan bigram dengan combiner: 411,217  (turun 1.8x)
Bigram unik: 349,114

10 bigram teratas (semua): [('virus corona', 2915), ('corona di', 2178), ('positif corona', 1804), ('di tengah', 1258), ('pandemi corona', 1075), ('di rumah', 1047), ('new normal', 917), ('di indonesia', 871), ('rp juta', 700), ('hari ini', 627)]

10 bigram teratas (tanpa stopword di kedua kata):
  virus corona            2,915
  positif corona          1,804
  pandemi corona          1,075
  new normal                917
  pasien corona             600
  cegah corona              583
  tengah pandemi            513
  kasus positif             442
  meninggal dunia           426
  gara gara                 424

'virus corona': 2,915


**Interpretasi:** pada judul dengan *n* kata terdapat *n − 1* bigram, sehingga jumlah pasangan **734.783** (= 825.601 − 90.818 judul). Setelah combiner turun menjadi **411.217** pasangan (turun hanya ± 1,8×, bandingkan ± 11× pada word count). Penurunannya kecil karena bigram jauh lebih beragam: ada **349.114 bigram unik**, kebanyakan hanya muncul sekali sehingga tidak ada yang bisa dijumlahkan di sisi mapper. Bigram teratas memperlihatkan frasa topik yang tidak terlihat pada kata tunggal: **`virus corona` (2.915)**, `positif corona` (1.804), `pandemi corona` (1.075), dan `new normal` (917).

## Latihan 4. Jumlah reducer
**Tugas:** jalankan H-11 dengan R = 2, 8, dan 32. Buat tabel beban reducer terberat dibanding bagian yang adil.

### Kode: tabel beban untuk R = 2, 8, 32

In [39]:
baris_tabel = []
for r in (2, 8, 32):
    b1, b2 = beban(hasil_map, r), beban(pasangan_combiner, r)
    adil = 100 / r
    p1, p2 = max(b1) / sum(b1) * 100, max(b2) / sum(b2) * 100
    baris_tabel.append({"R": r, "bagian adil (%)": round(adil, 2),
                        "terberat tanpa combiner (%)": round(p1, 2), "rasio thd adil": round(p1 / adil, 2),
                        "terberat dengan combiner (%)": round(p2, 2), "rasio (combiner)": round(p2 / adil, 2)})
tabel_reducer = pd.DataFrame(baris_tabel)
print(tabel_reducer.to_string(index=False))

 R  bagian adil (%)  terberat tanpa combiner (%)  rasio thd adil  terberat dengan combiner (%)  rasio (combiner)
 2            50.00                        52.27            1.05                         50.23              1.00
 8            12.50                        16.83            1.35                         12.85              1.03
32             3.12                         5.82            1.86                          3.44              1.10


**Interpretasi:** semakin banyak reducer, bagian adil mengecil (50% → 12,5% → 3,125%), tetapi reducer terberat **tidak turun secepat itu**: 52,27% (1,05× adil), 16,83% (1,35×), dan 5,82% (1,86×). Rasio terhadap bagian adil **naik** seiring R bertambah (tanpa combiner). Penyebabnya sama dengan H-11: satu key besar (`di` ± 3,3% pasangan) tidak bisa dipecah. Dengan combiner, setiap key jauh lebih kecil sehingga rasionya tetap dekat 1× (1,00×, 1,03×, 1,10×). Tabel ini melengkapi hasil R = 4, 16, 64 pada H-11.

## Latihan 5. Crawl ulang dengan jeda
**Tugas:** tambahkan jeda 0,05 detik pada `crawl_iter` untuk 50 halaman pertama. Berapa waktunya, dan berapa lama bila 911 halaman memakai jeda 1 detik?

### Kode: ukur waktu crawl 50 halaman, lalu ekstrapolasi

In [40]:
def waktu_crawl(n_hal, delay):
    t0 = time.perf_counter()
    n = sum(1 for _ in crawl_iter(BASE, range(1, n_hal + 1), delay=delay))
    return time.perf_counter() - t0, n

t_nol, n1 = waktu_crawl(50, 0.0)
t_005, n2 = waktu_crawl(50, 0.05)
print(f"50 halaman, jeda 0 detik   : {t_nol:6.2f} detik ({n1:,} judul)")
print(f"50 halaman, jeda 0,05 detik: {t_005:6.2f} detik ({n2:,} judul)")
print(f"Selisih                    : {t_005 - t_nol:6.2f} detik (teori: 50 x 0,05 = {50*0.05:.2f} detik)")

kerja_per_halaman = t_nol / 50                      # waktu kerja (unduh + parsing) per halaman
est_911 = N_HALAMAN * (kerja_per_halaman + 1.0)     # tambah jeda 1 detik per halaman
print(f"\nWaktu kerja per halaman     : {kerja_per_halaman*1000:.1f} ms")
print(f"Estimasi 911 halaman, jeda 1 detik: {est_911:,.0f} detik = {est_911/60:.1f} menit")
print(f"  (jeda saja: {N_HALAMAN*1.0:,.0f} detik = {N_HALAMAN/60:.1f} menit)")

50 halaman, jeda 0 detik   :   2.28 detik (5,000 judul)
50 halaman, jeda 0,05 detik:   5.07 detik (5,000 judul)
Selisih                    :   2.78 detik (teori: 50 x 0,05 = 2.50 detik)

Waktu kerja per halaman     : 45.7 ms
Estimasi 911 halaman, jeda 1 detik: 953 detik = 15.9 menit
  (jeda saja: 911 detik = 15.2 menit)


**Interpretasi:** jeda 0,05 detik menambah waktu sekitar **50 × 0,05 = 2,5 detik** pada 50 halaman (jumlah judul tetap 5.000). Bila 911 halaman memakai jeda 1 detik, waktunya ≈ **911 × (waktu kerja per halaman + 1 detik)**, yaitu sekitar **15–16 menit** (jeda saja sudah 911 detik ≈ 15,2 menit), dibandingkan puluhan detik tanpa jeda pada H-5. Inilah harga dari crawling yang sopan: jeda 1–2 detik adalah titik awal yang wajar pada situs sungguhan, dan itulah sebabnya H-5 memakai portal lokal dan H-4 membatasi diri pada 20 judul.

## Latihan 6. Sitemap lain
**Tugas:** ganti satu alamat di `SITEMAP_DIPILIH` dengan `sitemap_news` dari bagian lain yang tercantum di `sitemap.xml`. Kategori apa yang muncul? Terapkan `aturan_kualitas` dari H-6 pada 20 judul hasil crawl dan laporkan hasilnya.

### Kode: cari sitemap lain dari indeks, crawl satu sitemap saja, lalu terapkan aturan kualitas

> ⚠️ Sel ini menghubungi detik.com **sekali saja** (indeks + satu sitemap, jeda 2 detik). Dua sitemap pertama memakai ulang hasil H-4 (`sampel`) agar tidak mengulang request. Jangan dijalankan berulang. Bila server tidak terjangkau atau alamat indeks berbeda, sel memakai data contoh agar aturan kualitas tetap bisa diuji.

In [41]:
INDEKS_SITEMAP = "https://www.detik.com/sitemap.xml"   # ganti bila alamat indeks sitemap berbeda

sampel2, sumber_sampel2 = [], ""
try:
    sesi = requests.Session(); sesi.headers["User-Agent"] = UA_DETIK
    isi_indeks = BeautifulSoup(ambil_xml(INDEKS_SITEMAP, sesi), "xml")
    semua_loc = [x.get_text(strip=True) for x in isi_indeks.find_all("loc")]
    kandidat = [u for u in semua_loc if u.endswith("sitemap_news.xml")]
    host_dipakai = {urlparse(s).netloc for s in SITEMAP_DIPILIH}
    baru_url = next(u for u in kandidat if urlparse(u).netloc not in host_dipakai)
    print("Entri indeks:", len(semua_loc), "| sitemap_news:", len(kandidat))
    print("Alamat pengganti sitemap ke-3:", baru_url)
    dasar = sampel[:14] if len(sampel) >= 14 else []          # 7 news + 7 inet dari H-4
    baru = parse_sitemap_news(ambil_xml(baru_url, sesi))[:BATAS_TOTAL - len(dasar)]
    sampel2, sumber_sampel2 = dasar + baru, "crawl (H-4 + sitemap pengganti)"
except Exception as e:
    print("Crawl sitemap lain tidak dapat dilakukan:", type(e).__name__, "-", e)
    sampel2, sumber_sampel2 = parse_sitemap_news(CONTOH_XML), "data contoh CONTOH_XML (3 entri)"

d2 = pd.DataFrame(sampel2)
print(f"\nSumber data: {sumber_sampel2} | judul: {len(d2)}")
print("Kategori yang muncul:", d2["kategori"].value_counts().to_dict())
print("\nHasil aturan_kualitas (H-6):")
for nama, lulus in aturan_kualitas(d2).items():
    print(f"  {nama:<24} gagal: {(~lulus).sum():>3} dari {len(d2)}")

Entri indeks: 536 | sitemap_news: 248
Alamat pengganti sitemap ke-3: https://finance.detik.com/berita-ekonomi-bisnis/sitemap_news.xml

Sumber data: crawl (H-4 + sitemap pengganti) | judul: 20
Kategori yang muncul: {'news': 7, 'inet': 7, 'finance': 6}

Hasil aturan_kualitas (H-6):
  judul tidak kosong       gagal:   0 dari 20
  judul 3-30 kata          gagal:   0 dari 20
  kategori dikenal         gagal:   0 dari 20
  tanggal valid            gagal:   0 dari 20
  url artikel valid        gagal:   0 dari 20
  judul tidak duplikat     gagal:   0 dari 20


**Interpretasi:** kategori yang muncul ditentukan oleh subdomain sitemap pengganti (misalnya `finance`, `travel`, `oto`, `health`, `food` bila bagian itu yang terpilih), disertai `news` dan `inet` dari H-4. Aturan kualitas H-6 berlaku langsung pada hasil sitemap karena kolomnya sengaja diberi nama sama (`judul`, `kategori`, `tanggal`, `url`). Pada hasil crawl nyata, yang diharapkan: judul tidak kosong, 3–30 kata, kategori dikenal, tanggal valid, dan **url artikel valid** (karena URL sitemap berpola `/d-<angka>/`), sedangkan **judul tidak duplikat** bergantung pada isi hari itu. Berbeda dengan CSV, sumber sitemap tidak memiliki masalah `#`. **Isi hasilnya bervariasi tiap hari**, jadi catat angka dari output Anda sendiri pada laporan.

## Latihan 7. Kata kunci
**Tugas:** kolom `kata_kunci` berisi kata kunci tiap berita (dipisah koma). Tulis mapper dan reducer untuk menghitung kata kunci terbanyak pada hasil crawl Anda.

### Kode: mapper dan reducer kata kunci

In [42]:
import itertools

sumber_kw = sampel2 if sampel2 else parse_sitemap_news(CONTOH_XML)

def mapper_kw(rec):
    """Satu berita -> pasangan (kata_kunci, 1) untuk setiap kata kunci."""
    return [(k.strip().lower(), 1) for k in rec.get("kata_kunci", "").split(",") if k.strip()]

def reducer_kw(pasangan):
    """Masukan diurutkan dulu (shuffle), lalu dijumlahkan per key."""
    hasil_ = {}
    for kunci, grup in itertools.groupby(sorted(pasangan), key=lambda p: p[0]):
        hasil_[kunci] = sum(v for _, v in grup)
    return hasil_

pasangan_kw = [p for rec in sumber_kw for p in mapper_kw(rec)]
hasil_kw = reducer_kw(pasangan_kw)
print("Berita diproses :", len(sumber_kw), "| berita punya kata kunci:", sum(1 for r in sumber_kw if r["kata_kunci"]))
print("Pasangan kata kunci:", len(pasangan_kw), "| kata kunci unik:", len(hasil_kw))
print("\nKata kunci terbanyak:")
for k, n in collections.Counter(hasil_kw).most_common(10):
    print(f"  {k:<28}{n:>4}")

Berita diproses : 20 | berita punya kata kunci: 20
Pasangan kata kunci: 103 | kata kunci unik: 90

Kata kunci terbanyak:
  bareskrim polri                2
  bnr dana                       2
  dana                           2
  ekonomi indonesia              2
  karhutla                       2
  kecerdasan buatan              2
  kerugian negara                2
  korupsi                        2
  krisis ekonomi                 2
  mbg                            2


**Interpretasi:** mapper memecah `kata_kunci` pada koma dan mengeluarkan `(kata_kunci, 1)`; `sorted` berperan sebagai shuffle sehingga `reducer_kw` dapat menjumlahkan key yang berdampingan, persis pola reducer H-12. Karena hanya sebagian entri sitemap yang memuat kata kunci (pada contoh XML modul hanya entri pertama), hitungan didasarkan pada entri yang punya kolom itu saja. Dengan 20 judul hasil crawl, kebanyakan kata kunci hanya muncul sekali; peringkat teratas bervariasi tiap hari, jadi laporkan hasil crawl Anda sendiri.

---
# O. Tugas Mandiri: Inverted Index
**Tugas:** bangun indeks terbalik dengan MapReduce: untuk setiap kata, simpan daftar `id` judul yang memuatnya. Mapper menghasilkan `(kata, id)`, reducer menggabungkan id menjadi daftar. Kolom `id` kosong untuk kategori `inet`, tentukan sendiri cara menanganinya dan jelaskan alasannya.

### Kode: periksa kolom `id` dan putuskan penanganan baris `inet`

In [43]:
print("Baris dengan id kosong:", int(bersih["id"].isna().sum()), "| kategori:", bersih.loc[bersih["id"].isna(), "kategori"].unique().tolist())
print("Id kosong hanya pada inet:", set(bersih.loc[bersih["id"].isna(), "kategori"]) == {"inet"})
print("Id duplikat pada baris yang punya id:", int(bersih["id"].dropna().duplicated().sum()))

Baris dengan id kosong: 5577 | kategori: ['inet']
Id kosong hanya pada inet: True
Id duplikat pada baris yang punya id: 0


**Keputusan penanganan `inet`:** baris `inet` **tidak dibuang** (agar satu kategori utuh tetap ada pada indeks) dan **tidak diberi id palsu yang bisa tertukar dengan id asli**. Setiap baris `inet` diberi **id sintetis negatif** `-(nomor baris pada partisi inet + 1)`: unik, mudah dikenali (negatif = bukan id detik), dan tidak bertabrakan dengan id asli yang selalu positif. Karena hanya `inet` yang kosong (diverifikasi di atas), nomor baris per partisi cukup untuk menjamin keunikan. Pada daftar hasil, id asli ditampilkan lebih dulu, id sintetis di belakang.

### Kode: mapper, combiner, dan reducer inverted index

In [44]:
def mapper_indeks(path):
    """Mapper + combiner: satu split -> {kata: set(id)}; id kosong diganti id sintetis negatif."""
    df_ = pd.read_parquet(path, columns=["judul", "id"])
    lokal = collections.defaultdict(set)
    for nomor, (t, i) in enumerate(zip(df_["judul"], df_["id"])):
        id_ = int(i) if pd.notna(i) else -(nomor + 1)
        for kata in set(tokenisasi(t)):          # satu judul dihitung sekali per kata
            lokal[kata].add(id_)
    return lokal

def reducer_indeks(daftar_lokal):
    """Reducer: gabungkan id per kata; urut id asli (naik) lalu id sintetis."""
    gabungan = collections.defaultdict(set)
    for lokal in daftar_lokal:
        for kata, ids in lokal.items():
            gabungan[kata] |= ids
    return {kata: sorted(ids, key=lambda x: (x < 0, abs(x))) for kata, ids in gabungan.items()}

t0 = time.perf_counter()
indeks = reducer_indeks([mapper_indeks(p) for p in splits])
print(f"Indeks selesai dalam {time.perf_counter() - t0:.1f} detik | jumlah kata (key): {len(indeks):,}")
print(f"Total entri (kata, id) di indeks: {sum(len(v) for v in indeks.values()):,}")
print("\nTiga kata contoh beserta lima id pertamanya:")
for kata in ("corona", "jokowi", "banjir"):
    print(f"  {kata:<8} {len(indeks[kata]):>6,} judul | 5 id pertama: {indeks[kata][:5]}")

Indeks selesai dalam 1.5 detik | jumlah kata (key): 32,275
Total entri (kata, id) di indeks: 817,993

Tiga kata contoh beserta lima id pertamanya:
  corona   17,228 judul | 5 id pertama: [4856495, 4856598, 4868317, 4868545, 4869163]
  jokowi    2,316 judul | 5 id pertama: [4842674, 4842690, 4842703, 4842774, 4843215]
  banjir    1,992 judul | 5 id pertama: [4842198, 4842285, 4842286, 4842287, 4842292]


**Interpretasi:** setiap mapper memproses satu partisi dan langsung menggabungkan id per kata di sisi lokal (combiner berbentuk `set`), lalu reducer menyatukan hasil semua mapper. Hasilnya adalah indeks terbalik: kata → daftar id judul. Hasilnya: **32.275 kata (key)** dan 817.993 entri `(kata, id)`. Tiga kata contoh beserta lima id pertamanya (id asli diurutkan naik): `corona` (17.228 judul) → 4856495, 4856598, 4868317, 4868545, 4869163; `jokowi` (2.316 judul) → 4842674, 4842690, 4842703, 4842774, 4843215; `banjir` (1.992 judul) → 4842198, 4842285, 4842286, 4842287, 4842292.

### Kode: verifikasi indeks terhadap penghitungan langsung

In [45]:
# 1) Jumlah judul yang memuat kata = panjang daftar id (bandingkan dengan pencarian langsung pada DataFrame)
for kata in ("corona", "jokowi", "banjir"):
    langsung = int(bersih["judul"].map(lambda t: kata in set(tokenisasi(t))).sum())
    print(f"{kata:<8} panjang daftar id: {len(indeks[kata]):>6,} | hitung langsung: {langsung:>6,} | sama: {len(indeks[kata]) == langsung}")

# 2) Daftar id tidak boleh berisi duplikat, dan hitungan kata (H-9) >= jumlah judul (kata bisa muncul 2x dalam satu judul)
print("\nTanpa duplikat id:", all(len(v) == len(set(v)) for v in indeks.values()))
print("'corona' H-9 (kemunculan kata):", f"{hasil_semua['corona']:,}", ">= jumlah judul di indeks:", f"{len(indeks['corona']):,}")

# 3) Id sintetis hanya berasal dari kategori inet
n_sintetis = sum(1 for i in indeks["corona"] if i < 0)
print("Id sintetis (negatif) pada daftar 'corona':", n_sintetis, "| judul inet yang memuat corona:",
      int(bersih.loc[bersih["kategori"] == "inet", "judul"].map(lambda t: "corona" in set(tokenisasi(t))).sum()))

# Simpan sebagai contoh untuk lampiran (indeks lengkap disimpan ke siap_pakai/)
pd.DataFrame({"kata": list(indeks), "id_judul": list(indeks.values())}).to_parquet(SIAP / "inverted_index.parquet", index=False)
json.dump({k: indeks[k][:5] for k in ("corona", "jokowi", "banjir")}, open(SIAP / "contoh_inverted_index.json", "w"), indent=2)
print("\nTersimpan:", [p.name for p in sorted(SIAP.iterdir())])

corona   panjang daftar id: 17,228 | hitung langsung: 17,228 | sama: True
jokowi   panjang daftar id:  2,316 | hitung langsung:  2,316 | sama: True
banjir   panjang daftar id:  1,992 | hitung langsung:  1,992 | sama: True

Tanpa duplikat id: True
'corona' H-9 (kemunculan kata): 17,239 >= jumlah judul di indeks: 17,228
Id sintetis (negatif) pada daftar 'corona': 845 | judul inet yang memuat corona: 845

Tersimpan: ['contoh_inverted_index.json', 'inverted_index.parquet', 'kata_per_kategori.parquet', 'laporan.db']


**Interpretasi:** indeks terbukti benar: panjang daftar id untuk tiap kata **sama** dengan hasil hitung langsung, tidak ada id ganda, dan jumlah kemunculan kata pada H-9 selalu ≥ jumlah judul di indeks (sebuah kata dapat muncul lebih dari sekali dalam satu judul, tetapi satu judul hanya dicatat sekali). Id sintetis negatif hanya berasal dari baris `inet`. Indeks lengkap disimpan di `siap_pakai/inverted_index.parquet`, dan contoh tiga kata beserta lima id pertama disimpan di `siap_pakai/contoh_inverted_index.json` (untuk lampiran laporan).

---
# P. Pertanyaan Evaluasi

## P-1. Tiga tahap MapReduce
**Pertanyaan:** Jelaskan tiga tahap MapReduce dengan contoh kasus kantin atau word count. Pada tahap mana data berpindah antar-mesin?

### Kode: word count mini pada tiga split, dengan penanda perpindahan data

In [46]:
mini_split = {k: bersih.loc[bersih["kategori"] == k, "judul"].head(2).tolist() for k in ("finance", "health", "sport")}
R_MINI = 2

# MAP: tiap split (mesin) memproses datanya sendiri
keluaran_map = {k: [(w, 1) for j in js for w in tokenisasi(j)] for k, js in mini_split.items()}
for k, v in keluaran_map.items():
    print(f"MAP   split {k:<8}: {len(v):>2} pasangan, contoh {v[:3]}")

# SHUFFLE: pasangan dikirim ke reducer sesuai hash key  -> inilah tahap data berpindah antar-mesin
kotak = {r: collections.defaultdict(list) for r in range(R_MINI)}
for k, v in keluaran_map.items():
    for w, n in v:
        kotak[partitioner(w, R_MINI)][w].append(n)
total_pindah = sum(len(v) for v in keluaran_map.values())
print(f"\nSHUFFLE: {total_pindah} pasangan dikirim lewat jaringan ke {R_MINI} reducer ->",
      {r: sum(len(v) for v in d.values()) for r, d in kotak.items()})

# REDUCE: tiap reducer menjumlahkan nilai per key
for r, d in kotak.items():
    print(f"REDUCE reducer R{r}: ", dict(sorted((w, sum(v)) for w, v in d.items())))

MAP   split finance : 10 pasangan, contoh [('kemnaker', 1), ('awasi', 1), ('tka', 1)]
MAP   split health  : 26 pasangan, contoh [('pemuda', 1), ('sulsel', 1), ('curi', 1)]
MAP   split sport   : 16 pasangan, contoh [('foto', 1), ('pemain', 1), ('voli', 1)]

SHUFFLE: 52 pasangan dikirim lewat jaringan ke 2 reducer -> {0: 31, 1: 21}
REDUCE reducer R0:  {'ada': 1, 'andalkan': 1, 'bra': 2, 'cd': 2, 'curi': 2, 'di': 3, 'digitalkan': 1, 'dunia': 1, 'emak': 4, 'fetish': 2, 'gaji': 1, 'ilmu': 1, 'java': 1, 'jazz': 1, 'kontrakan': 1, 'pelatnas': 1, 'sulsel': 2, 'tak': 1, 'tka': 1, 'triady': 1, 'voli': 1}
REDUCE reducer R1:  {'atau': 2, 'awasi': 1, 'bni': 2, 'dan': 2, 'fauzi': 1, 'foto': 1, 'gangguan': 1, 'hitam': 1, 'jiwa': 1, 'kemnaker': 1, 'meikarta': 1, 'pantai': 1, 'pemain': 1, 'pemuda': 2, 'tercantik': 1, 'termasuk': 1, 'uang': 1}


**Interpretasi:** **Map** — setiap mapper memproses satu input split secara independen dan menghasilkan pasangan `(kata, 1)` (kantin: `(menu, jumlah)`). **Shuffle** — semua pasangan dengan key yang sama dikumpulkan ke reducer yang sama berdasarkan hash key; **inilah tahap ketika data berpindah antar-mesin** (lewat jaringan pada cluster), sehingga paling mahal. **Reduce** — setiap reducer menjumlahkan nilai per key (kantin: nasi goreng 5, es teh 8, mie ayam 4). Combiner opsional menjumlahkan dulu di sisi mapper agar yang berpindah pada shuffle lebih sedikit (12 → 9 pada kantin; 825.601 → 74.009 pada judul berita).

## P-2. Fungsi combiner dan kapan tidak boleh dipakai
**Pertanyaan:** Apa fungsi combiner, dan mengapa hasil akhir tidak berubah? Kapan combiner tidak boleh dipakai?

### Kode: combiner untuk penjumlahan (aman) vs rata-rata (salah bila naif)

In [47]:
# (a) Penjumlahan: asosiatif & komutatif -> combiner aman (bukti dari H-10 dan di bawah)
print("Word count dengan combiner sama persis dengan tanpa combiner:", dict(total) == hasil_semua)

# (b) Rata-rata: combiner yang langsung merata-ratakan SALAH
nilai_mapper = [[2, 4], [6]]                               # dua mapper; rata-rata sebenarnya = (2+4+6)/3 = 4
benar = sum(sum(m) for m in nilai_mapper) / sum(len(m) for m in nilai_mapper)
naif = sum(sum(m) / len(m) for m in nilai_mapper) / len(nilai_mapper)   # rata-rata dari rata-rata
# versi aman: combiner mengirim (jumlah, banyak), reducer yang membagi
aman_pasangan = [(sum(m), len(m)) for m in nilai_mapper]
aman = sum(s for s, _ in aman_pasangan) / sum(b for _, b in aman_pasangan)
print(f"\nRata-rata benar              : {benar}")
print(f"Combiner naif (rata-rata dari rata-rata): {naif}   <- SALAH")
print(f"Combiner aman (jumlah, banyak)          : {aman}   <- benar")

Word count dengan combiner sama persis dengan tanpa combiner: True

Rata-rata benar              : 4.0
Combiner naif (rata-rata dari rata-rata): 4.5   <- SALAH
Combiner aman (jumlah, banyak)          : 4.0   <- benar


**Interpretasi:** combiner adalah **reduce mini di sisi mapper**, dijalankan sebelum shuffle, untuk mengurangi data yang dikirim. Hasil akhir tidak berubah karena penjumlahan bersifat **asosiatif dan komutatif**: menjumlahkan sebagian dulu lalu menjumlahkan hasilnya sama dengan menjumlahkan sekaligus (terbukti `True`). Combiner **tidak boleh dipakai** bila operasi reduce tidak asosiatif/komutatif, atau bila hasil sebagian tidak bisa digabung begitu saja; contohnya **rata-rata** (rata-rata dari rata-rata ≠ rata-rata sebenarnya, 4,5 ≠ 4,0), median, atau persentil. Solusinya mengubah bentuk nilai antara (mis. kirim `(jumlah, banyak)`), dan perlu diingat Hadoop menganggap combiner sebagai optimasi yang boleh dijalankan nol, satu, atau beberapa kali sehingga program harus benar tanpa combiner.

## P-3. Input split dan pengaruh partisi
**Pertanyaan:** Apa yang dimaksud input split, dan bagaimana pembagian partisi di Praktikum 3 memengaruhi jumlah mapper?

### Kode: hitung split dan jumlah mapper menurut skema partisi

In [48]:
print("Input split pada lapisan_terkurasi (partisi kategori):", len(splits))
ukuran_split = {p.parent.name.split("=")[1]: len(pd.read_parquet(p, columns=["judul"])) for p in splits}
print(pd.Series(ukuran_split, name="judul per split").sort_values(ascending=False).to_string())
print(f"\nSplit terbesar / terkecil: {max(ukuran_split.values()):,} / {min(ukuran_split.values()):,} "
      f"= {max(ukuran_split.values())/min(ukuran_split.values()):.1f}x")

print("\nJumlah mapper menurut skema partisi (hasil L-5):")
for skema, r_ in ringkasan_partisi.items():
    print(f"  partisi {skema:<22} -> {r_['mapper (= jumlah file)']:>3} mapper")

Input split pada lapisan_terkurasi (partisi kategori): 9
news       32321
hot        16290
finance    14141
travel      6453
inet        5577
health      4914
oto         4375
food        4314
sport       2433

Split terbesar / terkecil: 32,321 / 2,433 = 13.3x

Jumlah mapper menurut skema partisi (hasil L-5):
  partisi ['kategori']           ->   9 mapper
  partisi ['bulan']              ->   6 mapper
  partisi ['kategori', 'bulan']  ->  54 mapper


**Interpretasi:** **input split** adalah potongan data yang diproses **satu mapper secara independen**; pada praktikum ini satu split = satu berkas partisi Parquet. Karena itu **cara data dipartisi menentukan jumlah mapper dan seberapa rata pekerjaan dibagi**: partisi `kategori` → 9 mapper, `bulan` → 6, `kategori + bulan` → 54. Pada partisi kategori ukuran split tidak seimbang (`news` ± 13× lebih besar daripada `sport`), sehingga mapper `news` menjadi yang paling lama (*straggler*) dan menentukan waktu selesai tahap map. Pada Hadoop sungguhan, split biasanya mengikuti ukuran blok HDFS, bukan kategori.

## P-4. Data skew
**Pertanyaan:** Apa itu data skew? Berikan contoh dari praktikum ini dan dua cara mengurangi dampaknya.

### Kode: contoh skew, lalu dua mitigasi (combiner dan key salting)

In [49]:
R_UJI = 64
heavy = {k for k, n in collections.Counter(hasil_semua).most_common(20)}   # 20 key terberat
GARAM = 8

def beban_garam(pasangan_per_mapper, r, garam=GARAM):
    """Key berat dipecah menjadi kata#0..kata#7 sebelum dipartisi; key lain tidak berubah."""
    b, urut, hasil_g = [0] * r, 0, collections.Counter()
    for h_ in pasangan_per_mapper:
        for kata, n in h_:
            kunci = f"{kata}#{urut % garam}" if kata in heavy else kata
            urut += 1
            b[zlib.crc32(kunci.encode()) % r] += 1
            hasil_g[kunci] += n
    return b, hasil_g

b0 = beban(hasil_map, R_UJI)
b_combiner = beban(pasangan_combiner, R_UJI)
b_garam, partial = beban_garam(hasil_map, R_UJI)
adil = 100 / R_UJI
print(f"Bagian adil pada {R_UJI} reducer: {adil:.2f}%")
print(f"1) Skew asli (tanpa combiner)        : terberat {max(b0)/sum(b0)*100:5.2f}%  ({max(b0)/sum(b0)*100/adil:.2f}x adil)")
print(f"2) Mitigasi 1: combiner              : terberat {max(b_combiner)/sum(b_combiner)*100:5.2f}%  ({max(b_combiner)/sum(b_combiner)*100/adil:.2f}x adil)")
print(f"3) Mitigasi 2: key salting (20 key berat, {GARAM} garam): terberat {max(b_garam)/sum(b_garam)*100:5.2f}%  ({max(b_garam)/sum(b_garam)*100/adil:.2f}x adil)")

# Tahap kedua: buang garam lalu jumlahkan -> hasil harus sama dengan H-9
akhir = collections.Counter()
for kunci, n in partial.items():
    akhir[kunci.split("#")[0]] += n
print("\nSetelah tahap kedua (buang garam, jumlahkan) sama dengan H-9:", dict(akhir) == hasil_semua)

Bagian adil pada 64 reducer: 1.56%
1) Skew asli (tanpa combiner)        : terberat  4.28%  (2.74x adil)
2) Mitigasi 1: combiner              : terberat  1.86%  (1.19x adil)
3) Mitigasi 2: key salting (20 key berat, 8 garam): terberat  2.25%  (1.44x adil)

Setelah tahap kedua (buang garam, jumlahkan) sama dengan H-9: True


**Interpretasi:** **data skew** adalah beban yang tidak merata antar-reducer, misalnya karena satu key sangat sering muncul sehingga reducer pemegang key itu menjadi *bottleneck*. **Contoh pada praktikum ini:** kata `di` = 3,3% seluruh pasangan sehingga pada 64 reducer reducer terberat menerima 4,28% (bagian adil 1,56%). **Dua cara mengurangi dampak:** (1) **combiner** — menjumlahkan lebih dulu di mapper sehingga setiap key hanya dikirim sekali per mapper, beban terdistribusi hampir rata; (2) **key salting** — key yang sangat berat diberi akhiran acak (`di#0` ... `di#7`) sehingga tersebar ke beberapa reducer, lalu tahap kedua membuang akhiran dan menjumlahkan (terbukti hasil akhirnya sama dengan H-9). Pada 64 reducer: skew asli 4,28% (2,74× adil), combiner 1,86% (1,19×), salting 2,25% (1,44×). Salting hanya menyebar 20 key terberat, jadi hasilnya sedikit di bawah combiner yang meratakan semua key. Cara lain yang umum: partitioner khusus atau menambah tahap agregasi bertingkat.

## P-5. Hadoop Streaming dan mapper Python
**Pertanyaan:** Mengapa Hadoop Streaming memungkinkan mapper ditulis dalam Python? Apa kelebihan dan kekurangannya dibanding mapper Java?

### Kode: lihat skrip streaming, lalu ukur ongkos teks lewat pipa dibanding Counter di dalam proses

In [50]:
print("=== mapper.py ===\n" + Path("mapper.py").read_text())
print("=== reducer.py ===\n" + Path("reducer.py").read_text())

# Data antar tahap hanya berupa TEKS di stdin/stdout -> bahasa apa pun bisa dipakai
keluaran_mapper = subprocess.run("python3 mapper.py < masukan.tsv", shell=True, capture_output=True, text=True, executable="/bin/bash").stdout
print("Keluaran mapper: ", f"{keluaran_mapper.count(chr(10)):,}", "baris teks,", f"{len(keluaran_mapper.encode())/1e6:.1f}", "MB")

t0 = time.perf_counter()
_ = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
t_dalam = time.perf_counter() - t0
t0 = time.perf_counter()
subprocess.run("cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > /dev/null", shell=True, executable="/bin/bash")
t_pipe = time.perf_counter() - t0
print(f"\nWord count di dalam satu proses Python (Counter): {t_dalam:6.2f} detik")
print(f"Word count lewat pipa streaming (teks + sort)   : {t_pipe:6.2f} detik  ({t_pipe/t_dalam:.0f}x)")

=== mapper.py ===
import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\n").split("\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\t1")

=== reducer.py ===
import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\n").split("\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\t{total}")

Keluaran mapper:  825,601 baris teks, 7.0 MB

Word count di dalam satu proses Python (Counter):   0.86 detik
Word count lewat pipa streaming (teks + sort)   :   2.33 detik  (3x)


**Interpretasi:** Hadoop Streaming hanya meminta mapper dan reducer **membaca `stdin` dan menulis `stdout`** (baris teks `key<TAB>value`). Hadoop tidak peduli bahasa pemrogramannya, sehingga Python, Bash, R, dan lain-lain dapat dipakai, dan skrip yang sama dapat diuji lebih dulu dengan pipe Linux `cat | mapper | sort | reducer`.
- **Kelebihan:** cepat menulis dan mudah dibaca; bisa memakai ekosistem Python (pandas, regex, dll.); mudah diuji lokal tanpa cluster; tidak perlu kompilasi dan pengemasan JAR.
- **Kekurangan:** semua data harus diubah menjadi **teks** dan di-*parse* kembali pada setiap tahap (ongkos serialisasi, terlihat pada selisih waktu di atas); tipe data hilang; start-up proses per tugas; kontrol terbatas atas fitur lanjutan Hadoop (mis. custom `Partitioner`, `InputFormat`, `Combiner` bertipe) yang lebih lengkap pada API Java; umumnya lebih lambat daripada mapper Java asli pada data besar.

## P-6. Flume: source, channel, sink
**Pertanyaan:** Apa peran source, channel, dan sink pada Flume? Mengapa channel membatasi ukuran antreannya?

### Kode: bandingkan channel berbatas dan tak berbatas ketika sink lebih lambat dari source

In [51]:
def uji_channel(maxsize, n=300, jeda_sink=0.002):
    q = queue.Queue(maxsize=maxsize)
    catat = {"maks": 0, "source_selesai": None}
    def src():
        t0 = time.perf_counter()
        for i in range(n):
            q.put(i)                                   # menunggu bila channel penuh (backpressure)
            catat["maks"] = max(catat["maks"], q.qsize())
        catat["source_selesai"] = time.perf_counter() - t0
        q.put(None)
    def snk():
        while q.get() is not None:
            time.sleep(jeda_sink)                      # sink lebih lambat daripada source
    ts, tk = threading.Thread(target=src), threading.Thread(target=snk)
    t0 = time.perf_counter(); ts.start(); tk.start(); ts.join(); tk.join()
    return catat["maks"], catat["source_selesai"], time.perf_counter() - t0

print(f"{'channel':<22}{'antrean maks':>14}{'source selesai (s)':>20}{'total (s)':>11}")
for nama, ms in (("berbatas (maxsize=10)", 10), ("tak berbatas", 0)):
    maks, t_src, t_total = uji_channel(ms)
    print(f"{nama:<22}{maks:>14}{t_src:>20.3f}{t_total:>11.3f}")

channel                 antrean maks  source selesai (s)  total (s)
berbatas (maxsize=10)             10               0.611      0.634
tak berbatas                     300               0.000      0.633


**Interpretasi:**
- **Source** menerima/mengambil data dari sumber (di praktikum ini: crawler), **channel** adalah antrean sementara yang menjembatani, dan **sink** menulis ke tujuan (berkas JSONL di `lapisan_mentah/`). Channel membuat pengirim dan penulis dapat bekerja dengan **kecepatan berbeda**.
- Pada percobaan, sink sengaja dibuat lebih lambat. Pada channel **berbatas** antrean tidak pernah melebihi batas dan source **terpaksa menunggu** (*backpressure*, waktu source selesai ≈ waktu sink). Pada channel **tak berbatas** source selesai hampir seketika dan seluruh data menumpuk di memori (antrean maksimum mendekati jumlah event). Karena itulah channel dibatasi: untuk **melindungi memori** dan memaksa source menyesuaikan laju dengan sink (di H-5 batasnya 2.000 event).

## P-7. Sqoop import dan export
**Pertanyaan:** Apa yang dilakukan Sqoop import dan export? Mengapa import memakai beberapa mapper dengan rentang kunci?

### Kode: simulasi `sqoop import` dengan 4 mapper pada rentang `rowid`

In [52]:
from concurrent.futures import ThreadPoolExecutor

DB_PATH = SIAP / "laporan.db"
koneksi = sqlite3.connect(DB_PATH)
lo, hi = koneksi.execute("SELECT MIN(rowid), MAX(rowid) FROM kata_per_kategori").fetchone()
total_db = koneksi.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0]
koneksi.close()

N_MAPPER = 4
lebar = math.ceil((hi - lo + 1) / N_MAPPER)
rentang = [(lo + i * lebar, min(lo + (i + 1) * lebar - 1, hi)) for i in range(N_MAPPER)]
print("Kolom pembagi (--split-by): rowid | rentang tiap mapper:", rentang)

IMPOR = Path("impor_sqoop"); shutil.rmtree(IMPOR, ignore_errors=True); IMPOR.mkdir()

def mapper_impor(args):
    nomor, (a, b) = args
    kon = sqlite3.connect(DB_PATH)                      # satu koneksi per mapper
    baris_ = kon.execute("SELECT kategori, kata, jumlah FROM kata_per_kategori WHERE rowid BETWEEN ? AND ?", (a, b)).fetchall()
    kon.close()
    pd.DataFrame(baris_, columns=["kategori", "kata", "jumlah"]).to_csv(IMPOR / f"part-m-{nomor:05d}.csv", index=False)
    return len(baris_)

with ThreadPoolExecutor(N_MAPPER) as ex:
    n_per_mapper = list(ex.map(mapper_impor, enumerate(rentang)))

print("Baris per mapper:", n_per_mapper, "| jumlah:", f"{sum(n_per_mapper):,}", "| baris di tabel:", f"{total_db:,}")
kembali = pd.concat([pd.read_csv(p, keep_default_na=False) for p in sorted(IMPOR.glob("part-m-*.csv"))], ignore_index=True)
print("Berkas hasil:", [p.name for p in sorted(IMPOR.glob('part-m-*.csv'))])
print("Impor lengkap dan tanpa duplikat:", len(kembali) == total_db and
      kembali.merge(hasil_df, on=["kategori", "kata", "jumlah"]).shape[0] == total_db)

Kolom pembagi (--split-by): rowid | rentang tiap mapper: [(1, 18401), (18402, 36802), (36803, 55203), (55204, 73601)]
Baris per mapper: [18401, 18401, 18401, 18398] | jumlah: 73,601 | baris di tabel: 73,601
Berkas hasil: ['part-m-00000.csv', 'part-m-00001.csv', 'part-m-00002.csv', 'part-m-00003.csv']
Impor lengkap dan tanpa duplikat: True


**Interpretasi:** **Sqoop** memindahkan data besar antara **Hadoop dan database relasional**. *Import* membawa tabel dari database ke Hadoop (HDFS); *export* membawa hasil pemrosesan Hadoop kembali ke tabel database (pada H-13 kita mensimulasikan arah export dengan `to_sql`). Import memakai **beberapa mapper dengan rentang kunci** (`--split-by`): rentang nilai kolom kunci dibagi menjadi bagian-bagian yang tidak tumpang tindih, dan tiap mapper membaca satu rentang secara **paralel**. Rentang yang saling lepas menjamin tidak ada baris yang terbaca dua kali atau terlewat, sementara paralelisme mempercepat transfer; pada percobaan, empat mapper membaca rentang `rowid` berbeda dan jumlahnya sama dengan seluruh baris tabel (73.601).

## P-8. Peran HDFS dan YARN
**Pertanyaan:** Sebutkan peran HDFS dan YARN. Di bagian mana dari praktikum ini peran keduanya terlihat, dan di bagian mana hanya disimulasikan?

### Kode: simulasi pemecahan blok (HDFS) dan penjadwalan tugas ke slot (YARN)

In [53]:
# HDFS (simulasi): berkas dipecah menjadi blok dan direplikasi di banyak mesin
ukuran_berkas = os.path.getsize("masukan.tsv")
for blok_mb in (128, 1):
    n_blok = math.ceil(ukuran_berkas / (blok_mb * 1024 ** 2))
    print(f"masukan.tsv = {ukuran_berkas/1e6:.1f} MB | blok {blok_mb:>3} MB -> {n_blok} blok "
          f"| dengan replikasi 3x: {n_blok*3} salinan blok")

# YARN (simulasi): 9 tugas map dijadwalkan ke 2 "slot" (proses pekerja); sisanya mengantre
def tugas_map(p):
    return (p.parent.name.split("=")[1], os.getpid(), len(pd.read_parquet(p, columns=["judul"])))

with Pool(2) as pool:
    penjadwalan = pool.map(tugas_map, splits)
pekerja = {pid: f"slot-{i+1}" for i, pid in enumerate(sorted({pid for _, pid, _ in penjadwalan}))}
print(f"\n{'tugas map (split)':<20}{'dijalankan di':<15}{'judul':>8}")
for kat, pid, n in penjadwalan:
    print(f"{kat:<20}{pekerja[pid]:<15}{n:>8,}")

masukan.tsv = 6.1 MB | blok 128 MB -> 1 blok | dengan replikasi 3x: 3 salinan blok
masukan.tsv = 6.1 MB | blok   1 MB -> 6 blok | dengan replikasi 3x: 18 salinan blok

tugas map (split)   dijalankan di     judul
finance             slot-1           14,141
food                slot-1            4,314
health              slot-2            4,914
hot                 slot-2           16,290
inet                slot-1            5,577
news                slot-1           32,321
oto                 slot-2            4,375
sport               slot-2            2,433
travel              slot-2            6,453


**Interpretasi:**
- **HDFS** menyimpan berkas besar dengan memecahnya menjadi **blok** (default 128 MB) yang tersebar dan direplikasi (default 3×) di banyak mesin, sehingga data tahan terhadap kegagalan mesin dan tiap blok dapat diproses paralel. **YARN** mengatur sumber daya (CPU, memori) di cluster dan menjalankan job: ia menerima permintaan dan membagikan tugas ke *container* di mesin-mesin yang tersedia. MapReduce hanyalah salah satu program yang berjalan di atas YARN.
- **Pada praktikum ini keduanya hanya disimulasikan:** pemecahan data menjadi partisi/split Parquet meniru gagasan blok HDFS (percobaan di atas menunjukkan berkas 6 MB hanya 1 blok pada ukuran 128 MB, sehingga **MapReduce baru bermanfaat untuk data yang tidak muat di satu mesin**), dan `multiprocessing.Pool` meniru YARN yang membagi 9 tugas ke 2 slot. Hadoop sungguhan dengan HDFS dan YARN baru terlihat bila bagian **opsional H-12** (local mode, tanpa HDFS dan YARN) atau cluster betulan dijalankan.

---
# Q. Kesimpulan

Pipeline data tidak berhenti pada satu alat: **ingestion** menentukan apa yang masuk, **quality** menentukan apa yang bisa dipercaya, **storage** menentukan seberapa cepat data dibaca, dan **pemrosesan** menentukan apa yang bisa dijawab. Praktikum ini memperlihatkan satu alur dari halaman web sampai laporan, dan tiga pelajaran MapReduce yang dibuktikan dengan kode:

1. **Shuffle itu mahal, sehingga combiner sangat bernilai**: pasangan yang dikirim turun dari 825.601 menjadi 74.009 tanpa mengubah hasil (H-10, L-1).
2. **Paralel pada data kecil bisa lebih lambat**, karena biaya koordinasi dan perpindahan data melampaui kerja yang dibagi (H-10, L-1).
3. **Skew berasal dari data, bukan dari jumlah mesin**: menambah reducer tidak menolong bila satu key menanggung beban besar (H-11, L-2, P-4).

## Penyimpanan hasil dan pengumpulan
### Kode: zip folder untuk dikumpulkan (dan salin ke Google Drive bila di Colab)

> Yang dikumpulkan: (1) notebook yang sudah dijalankan; (2) folder `lapisan_mentah/`, `lapisan_terkurasi/`, `siap_pakai/` (zip, bila terlalu besar cukup contoh); (3) laporan 2–3 halaman berisi jawaban Analisis Hasil, studi kasus, dan hasil inverted index (tiga kata contoh beserta lima id pertamanya).

In [54]:
for folder in ("lapisan_mentah", "lapisan_terkurasi", "siap_pakai"):
    arsip = shutil.make_archive(folder, "zip", folder)
    print(f"{arsip:<28}{os.path.getsize(arsip)/1e6:>6.1f} MB")

# Salin ke Google Drive (hanya berjalan di Colab; dilewati bila bukan Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    tujuan = Path("/content/drive/MyDrive/BigData/Praktikum5")
    tujuan.mkdir(parents=True, exist_ok=True)
    for nama in ("lapisan_mentah.zip", "lapisan_terkurasi.zip", "siap_pakai.zip"):
        shutil.copy(nama, tujuan / nama)
    print("Tersalin ke", tujuan)
except ImportError:
    print("Bukan Colab: salinan ke Google Drive dilewati.")

/content/lapisan_mentah.zip    5.1 MB
/content/lapisan_terkurasi.zip   7.2 MB
/content/siap_pakai.zip        3.5 MB
Mounted at /content/drive
Tersalin ke /content/drive/MyDrive/BigData/Praktikum5


**Interpretasi:** tiga lapisan data dikemas menjadi berkas zip untuk dikumpulkan. Di Colab, sel yang sama juga menyalin zip ke `MyDrive/BigData/Praktikum5` agar tidak hilang saat sesi berakhir. Simpan juga notebook ini (`File → Save a copy in Drive`) setelah seluruh sel dijalankan.

---
# (Opsional) Hadoop Streaming Asli
Bagian opsional dari H-12 / nilai tambah: jalankan `mapper.py` dan `reducer.py` yang sama pada **Hadoop 3.3.6 local mode**. Sel di bawah mengunduh Hadoop (± 700 MB) dan memasang Java, sehingga dapat memakan beberapa menit. Letaknya di akhir agar tidak menghentikan *Run all*. Lampirkan log dan hasil `diff` bila dikerjakan.

In [55]:
%%bash
sudo apt-get install -y openjdk-11-jdk-headless
wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
tar -xzf hadoop-3.3.6.tar.gz
export HADOOP_HOME=$PWD/hadoop-3.3.6
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
rm -rf keluaran_hadoop
$HADOOP_HOME/bin/hadoop jar \
  $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input masukan.tsv -output keluaran_hadoop
ls keluaran_hadoop
diff <(LC_ALL=C sort hasil_pipe.txt) keluaran_hadoop/part-00000 && echo "SAMA"

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  openjdk-11-jre-headless
Suggested packages:
  openjdk-11-demo openjdk-11-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-11-jdk-headless openjdk-11-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 0 not upgraded.
Need to get 116 MB of archives.
After this operation, 258 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 openjdk-11-jre-headless amd64 11.0.32.1+1-1ubuntu1~24.04 [42.3 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 openjdk-11-jdk-headless amd64 11.0.32.1+1-1ubuntu1~24.04 [73.7 MB]
Fetched 116 MB in 2s (51.2 MB/s)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 1

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
2026-10-30 01:53:59,936 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-10-30 01:54:00,304 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-10-30 01:54:00,304 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-10-30 01:54:00,354 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-10-30 01:54:00,706 INFO mapred.FileInputFormat: Total input files to process : 1
2026-10-30 01:54:00,733 INFO mapreduce.JobSubmitter: number of splits:1

**Interpretasi:** mapper dan reducer yang sama dijalankan pada Hadoop 3.3.6 *local mode* (satu mesin, tanpa HDFS dan YARN). Bila hasilnya benar, `diff` mencetak `SAMA`, artinya keluaran Hadoop asli identik dengan hasil pipe.